In [ ]:
import os
print(os.listdir("/kaggle/input/datasets/hearfool/vggface2"))


In [ ]:
import os
import shutil
import random
from pathlib import Path

src = "/kaggle/input/datasets/hearfool/vggface2/train"
dst = "/kaggle/working/faces_by_identity_vgg"

# v_C_vgg — GIỮ NGUYÊN scale từ bản gốc (2026-08-16): 1500 identity, để
# thống nhất với bài paper (MUFAC Hướng C dùng cùng scale tương đối lớn).
N_IDENTITIES      = 1500  # giảm từ 3000 — tổng pipeline (Original+baseline+LPEU+repair) phải vừa 1 session Kaggle
MIN_IMAGES        = 20
MAX_IMAGES_PER_ID = 40    # giảm so với 60 để bù lại việc tăng N_IDENTITIES 10x, tránh nổ thời gian train

random.seed(42)
os.makedirs(dst, exist_ok=True)

# VGGFace2 đã chia sẵn theo folder-per-identity (n000002/, n000003/...)
# => KHÔNG tách identity từ filename như Korean Family nữa.
all_ids = sorted(os.listdir(src))
eligible = [
    pid for pid in all_ids
    if os.path.isdir(os.path.join(src, pid))
    and len(os.listdir(os.path.join(src, pid))) >= MIN_IMAGES
]
print(f"Identity đủ điều kiện (>= {MIN_IMAGES} ảnh): {len(eligible)}")

random.shuffle(eligible)
selected = eligible[:N_IDENTITIES]

for pid in selected:
    src_dir = os.path.join(src, pid)
    dst_dir = os.path.join(dst, pid)
    os.makedirs(dst_dir, exist_ok=True)

    imgs = [f for f in os.listdir(src_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    random.shuffle(imgs)
    for img in imgs[:MAX_IMAGES_PER_ID]:
        shutil.copy(os.path.join(src_dir, img), os.path.join(dst_dir, img))

print(f"Done converting dataset — đã chọn {len(selected)} identity → {dst}")


In [ ]:
root = "/kaggle/working/faces_by_identity_vgg"

counts = {}

for person in os.listdir(root):
    p = os.path.join(root, person)
    if os.path.isdir(p):
        counts[person] = len(os.listdir(p))

print("num identities:", len(counts))
print("min:", min(counts.values()))
print("max:", max(counts.values()))


In [ ]:
import os, math, json, copy, random
from dataclasses import dataclass, asdict, field
from typing import Dict, List, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, models, transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


In [ ]:
# import os
# cache = "./runs/lpeu_v7_vggface2_C/original_model.pt"
# if os.path.exists(cache):
#     os.remove(cache)
#     print("Cache deleted")
# else:
#     print("No cache — will train fresh")


In [ ]:
# ===========================================================================
# ▓  CELL B: CONFIG
# ===========================================================================
#
# v_C_vgg (2026-08-19) — PORT của lpeu-v7-mufac-person-C.ipynb (Hướng C:
# Migrate→Shatter, checkpoint-selection fix, gradient projection, L_div)
# sang VGGFace2. Khác biệt cấu trúc quan trọng nhất so với MUFAC:
#
#   MUFAC: identity (family/person) ≠ classifier class (age, 8 lớp) —
#          identity_equals_class=False → TOÀN BỘ đầu ArcFace bị đóng băng
#          mỗi forget-step → forget_accuracy KHÔNG THỂ đổi (tính chất kiến
#          trúc, không phải bug — xem companion paper).
#   VGGFace2: identity CHÍNH LÀ classifier class (1 folder = 1 identity =
#          1 lớp ArcFace) — identity_equals_class=True → chỉ các hàng
#          ArcFace RETAIN bị đóng băng/khôi phục, hàng FORGET được tự do
#          thay đổi trong forget-step (và bị đóng băng lại đúng ở giá trị
#          SAU forget trong lúc repair) → forget_accuracy CÓ THỂ giảm thật.
#          Cơ chế identity_equals_class() (Cell F) đã tổng quát hoá sẵn cho
#          cả 2 trường hợp — KHÔNG cần sửa gì ở run_lpeu_v7_unlearning /
#          run_retain_repair, chỉ cần forget_ids là int trong [0,num_classes).
#
# Các trường forget_manual_lr / repair_layer4_lr_mult / repair_r2_kd_mult
# GIỮ NGUYÊN giá trị đã tune riêng cho VGGFace2 ở bản gốc (16/8) — các cơ
# chế MỚI (Migrate→Shatter, checkpoint combined-balance, gradient
# projection, repair catch-up, L_div) chưa từng được tune thực nghiệm trên
# VGGFace2 (chỉ mới qua nhiều vòng tune trên MUFAC — xem lịch sử v7.4-v7.9
# trong lpeu-v7-mufac-person-C.ipynb) — các ngưỡng liên quan (migrate_*,
# shatter_erase_floor, checkpoint_retain_gate, repair_catchup_*, w_div) giữ
# nguyên default đã tune trên MUFAC làm điểm khởi đầu HỢP LÝ, nhưng CẦN đọc
# log epoch đầu tiên thực chạy trên VGGFace2 để kiểm tra/chỉnh lại — xem
# CELL N (Tuning Guide) ở cuối notebook.

@dataclass
class Config:
    # ── paths ─────────────────────────────────────────────────────────────
    dataset_root:  str = "./faces_by_identity_vgg"
    output_dir:    str = "./runs/lpeu_v7_vggface2_C"
    seed:          int = 42
    device:        str = "cuda" if torch.cuda.is_available() else "cpu"

    # ── data ──────────────────────────────────────────────────────────────
    image_size:               int   = 128
    batch_size:               int   = 32
    num_workers:              int   = 2
    min_images_per_identity:  int   = 8
    train_ratio:              float = 0.70
    val_ratio:                float = 0.10
    test_ratio:               float = 0.20
    # tỉ lệ identity giữ lại HOÀN TOÀN ngoài train/val/test, chưa từng được
    # model gốc nhìn thấy — cần thiết để tính đúng Forgetting Score / MIA
    # kiểu Choi & Na (2023) / GLI (Choi et al. 2024): so forget vs UNSEEN,
    # KHÔNG phải forget vs retain-test.
    unseen_ratio:             float = 0.10

    # ── model ─────────────────────────────────────────────────────────────
    embedding_dim:       int   = 128
    pretrained_backbone: bool  = True
    arcface_s:           float = 32.0
    arcface_m:           float = 0.30

    # ── original training ─────────────────────────────────────────────────
    original_epochs:      int   = 120
    original_lr:          float = 0.05
    original_momentum:    float = 0.9
    original_wd:          float = 1e-4
    warmup_epochs:        int   = 5
    early_stop_patience:  int   = 20
    lr_reduce_patience:   int   = 4

    # ── dataset splits ────────────────────────────────────────────────────
    # v_C_vgg — TĂNG 30 → 100, thống nhất SỐ LƯỢNG identity bị quên với
    # MUFAC Hướng C (num_forget_ids=100) để 2 bài có thể so sánh trực tiếp
    # trong paper (khác tỉ lệ % — 100/1500 ≈ 6.7% ở đây so với 100/809 ≈
    # 12% ở MUFAC — nhưng SỐ TUYỆT ĐỐI identity bị quên bằng nhau).
    num_forget_ids: int = 100
    neighbor_k:     int = 64

    # ── staged unlearning schedule (anchor phases) ───────────────────────
    stage1_epochs: int = 3    # Phase 1: anchor OFF
    stage2_epochs: int = 20   # Phase 2: anchor 50%
    stage3_epochs: int = 8    # Phase 3: anchor 100%

    # ── LR ────────────────────────────────────────────────────────────────
    unlearn_lr:      float = 5e-6
    unlearn_wd:      float = 1e-6
    layer4_lr_mult:  float = 0.01

    # v_C_vgg — GIỮ NGUYÊN giá trị đã tune riêng cho VGGFace2 (bản gốc
    # 16/8), KHÁC với MUFAC (1e-3) — hai dataset có thang gradient khác
    # nhau (xem ghi chú v7.2 gốc: forget step dùng SGD thủ công, bypass
    # AdamW, để boost không bị chuẩn hoá triệt tiêu).
    forget_manual_lr: float = 2e-4

    # ── retain path ───────────────────────────────────────────────────────
    w_kd_global:    float = 30.0
    kd_temperature: float = 2.0
    k_anchor:       int   = 30
    w_kd_local:     float = 15.0
    use_ewc:        bool  = True
    w_ewc:          float = 50.0
    w_ce_retain:    float = 5.0

    # ── RETAIN-PROTOTYPE ANCHOR ──────────────────────────────────────────
    use_proto_retain: bool  = True
    w_proto_retain:   float = 20.0

    # ── IN-LOOP REPAIR ────────────────────────────────────────────────────
    use_inloop_repair: bool  = True
    w_repair:          float = 20.0

    # ── POST-UNLEARNING REPAIR (2-phase + R3 catch-up) ──────────────────
    use_post_repair:  bool  = True
    repair_epochs:    int   = 20
    repair_lr:        float = 5e-5
    # v_C_vgg — GIỮ NGUYÊN giá trị VGGFace2-gốc (khác MUFAC: 0.35/0.15) —
    # chưa có bằng chứng thực nghiệm mới để đổi.
    repair_layer4_lr_mult: float = 0.25
    repair_r2_kd_mult:     float = 1.0

    # v_C_vgg — PORT nguyên khối từ MUFAC-C (v7.4-v7.9), CHƯA tune riêng
    # cho VGGFace2 — xem Tuning Guide (Cell N) trước khi chạy lần đầu.
    repair_forget_guard_weight: float = 1.5
    repair_catchup_enable:      bool  = True
    repair_catchup_target:      float = 1.00
    repair_catchup_max_epochs:  int   = 30

    # ── forget path ───────────────────────────────────────────────────────
    w_erase:       float = 2.0
    w_kl_uniform:  float = 20.0
    w_repel:       float = 4.0
    repel_margin:  float = -0.3

    # ── Hướng C: L_div (PIU-FR-style KL-to-unseen) ──────────────────────
    # v_C_vgg — Ở MUFAC, L_div dùng "class-matched" (unseen cùng age-class
    # với ảnh forget) vì age-class là nhãn THÔ dùng chung giữa forget và
    # unseen. Ở VGGFace2, identity CHÍNH LÀ class, và D_unseen gồm các
    # identity TÁCH BIỆT HOÀN TOÀN (disjoint) với identity forget/retain —
    # không có nhãn chung nào để "khớp lớp". Do đó L_div ở đây lấy target
    # là phân phối CỦA MODEL GỐC trên một batch lấy NGẪU NHIÊN (không phân
    # biệt lớp) từ TOÀN BỘ D_unseen — xem _sample_unseen_batch_vgg (Cell
    # I). Ý nghĩa cốt lõi vẫn giữ: kéo hành vi model-forget trên ảnh forget
    # về đúng hành vi model GỐC trên dữ liệu THẬT SỰ chưa từng thấy — chỉ
    # khác là không thể "class-matched" như MUFAC.
    w_div:                 float = 20.0
    div_unseen_pool_max:   int   = 2000   # trần số ảnh unseen giữ trong RAM cho L_div

    # ── MIGRATE → SHATTER (2-stage forget loss) ─────────────────────────
    migrate_target_fraction: float = 0.35
    migrate_max_epochs:      int   = 10
    shatter_erase_floor:     float = 0.18

    # ── PCGrad ────────────────────────────────────────────────────────────
    use_pcgrad:   bool  = True
    pcgrad_boost: float = 40.0

    # ── Gradient projection (v7.5) ───────────────────────────────────────
    use_grad_projection: bool = True

    # ── clipping ──────────────────────────────────────────────────────────
    grad_clip_retain: float = 0.1
    grad_clip_forget: float = 2.0

    # ── safety ────────────────────────────────────────────────────────────
    retain_acc_floor:        float = 0.90
    checkpoint_retain_gate:  float = 0.85
    safety_grace_epochs:     int   = 3
    safety_patience:         int   = 2

    # ── MIA ───────────────────────────────────────────────────────────────
    mia_n_members:    int = 300
    mia_n_nonmembers: int = 300

    unlearn_epochs: int = 0   # tính tự động trong __post_init__

    def __post_init__(self):
        self.unlearn_epochs = self.stage1_epochs + self.stage2_epochs + self.stage3_epochs


cfg = Config()
os.makedirs(cfg.output_dir, exist_ok=True)

def set_seed(seed: int):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)

print(f"{'─'*65}")
print(f"  LPEU-VGGFace2 (Hướng C)  |  Device: {cfg.device}")
print(f"{'─'*65}")
print(f"  Model:  emb_dim={cfg.embedding_dim}  s={cfg.arcface_s}  m={cfg.arcface_m}  [Simple Neck]")
print(f"  Staged unlearning: Phase1={cfg.stage1_epochs}e / Phase2={cfg.stage2_epochs}e / Phase3={cfg.stage3_epochs}e")
print(f"  EWC:    {'ON' if cfg.use_ewc else 'OFF'}  w_ewc={cfg.w_ewc}")
print(f"  Forget: w_erase={cfg.w_erase}  w_repel={cfg.w_repel}  w_div={cfg.w_div}  margin={cfg.repel_margin}  boost={cfg.pcgrad_boost}×")
print(f"  Forget manual LR (bypass AdamW): {cfg.forget_manual_lr:.1e}")
print(f"  num_forget_ids={cfg.num_forget_ids}")
print(f"{'─'*65}")


In [ ]:
# ===========================================================================
# ▓  CELL C: DATASET UTILITIES (VGGFace2)
# ===========================================================================

class IndexedSubset(Dataset):
    """Wraps a dataset; returns (x, y, original_idx)."""
    def __init__(self, dataset, indices):
        self.dataset = dataset
        self.indices = list(indices)
    def __len__(self):  return len(self.indices)
    def __getitem__(self, i):
        x, y = self.dataset[self.indices[i]]
        return x, y, self.indices[i]


def build_transforms(image_size: int):
    train_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(0.2, 0.2, 0.2),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((image_size, image_size)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return train_tf, eval_tf


def make_loader(dataset, batch_size, shuffle, num_workers=2):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle,
                      num_workers=num_workers, pin_memory=True)


def load_full_dataset(cfg: Config):
    train_tf, eval_tf = build_transforms(cfg.image_size)
    base = datasets.ImageFolder(cfg.dataset_root)
    class_to_indices: Dict[int, List[int]] = {}
    for idx, (_, y) in enumerate(base.samples):
        class_to_indices.setdefault(y, []).append(idx)
    valid_classes = {c for c, idxs in class_to_indices.items()
                     if len(idxs) >= cfg.min_images_per_identity}
    filtered = [i for i, (_, y) in enumerate(base.samples) if y in valid_classes]
    train_base = datasets.ImageFolder(cfg.dataset_root, transform=train_tf)
    eval_base  = datasets.ImageFolder(cfg.dataset_root, transform=eval_tf)
    return train_base, eval_base, filtered, sorted(valid_classes)


def split_by_identity(dataset, filtered_indices, cfg: Config):
    """v7.7 — THÊM D_unseen: trước khi chia train/val/test, trích riêng một
    số danh tính (unseen_ratio, mặc định 10%) giữ HOÀN TOÀN ngoài mọi giai
    đoạn train — chưa từng được model gốc nhìn thấy. Cần thiết để tính đúng
    Forgetting Score / MIA kiểu Choi & Na (2023) / GLI (Choi et al. 2024):
    hai công trình đó so loss của x_forget với loss của x_unseen (KHÔNG
    phải x_retain_test).

    VGGFace2: identity CHÍNH LÀ classifier class — forget_ids trả về là
    CÁC SỐ NGUYÊN trong [0, num_classes), khác MUFAC (family-id dạng str)
    — đây là điều kiện để identity_equals_class() (Cell F) nhận diện đúng
    và kích hoạt cơ chế "chỉ đóng băng hàng ArcFace retain" thay vì đóng
    băng toàn bộ đầu."""
    class_to_indices: Dict[int, List[int]] = {}
    for idx in filtered_indices:
        _, y = dataset.samples[idx]
        class_to_indices.setdefault(y, []).append(idx)

    all_classes = sorted(class_to_indices.keys())
    random.seed(cfg.seed)
    shuffled_classes = all_classes.copy()
    random.shuffle(shuffled_classes)
    n_unseen = max(1, int(len(shuffled_classes) * cfg.unseen_ratio))
    unseen_classes = set(shuffled_classes[:n_unseen])
    trainable_classes = [c for c in all_classes if c not in unseen_classes]

    unseen_idx = []
    for cls in unseen_classes:
        unseen_idx.extend(class_to_indices[cls])

    train_idx, val_idx, test_idx = [], [], []
    eligible_forget = []

    for cls in trainable_classes:
        idxs = class_to_indices[cls]
        random.shuffle(idxs)
        n = len(idxs)
        n_tr = max(1, int(n * cfg.train_ratio))
        n_va = max(1, int(n * cfg.val_ratio))
        n_te = max(1, n - n_tr - n_va)
        if n_tr + n_va + n_te > n:
            n_tr = max(1, n_tr - 1)
        train_idx.extend(idxs[:n_tr])
        val_idx.extend(idxs[n_tr:n_tr+n_va])
        test_idx.extend(idxs[n_tr+n_va:n_tr+n_va+n_te])
        if n_tr >= 2: eligible_forget.append(cls)

    random.seed(cfg.seed)
    forget_ids = random.sample(eligible_forget, cfg.num_forget_ids)
    fset = set(forget_ids)

    forget_idx, retain_train_idx = [], []
    for idx in train_idx:
        _, y = dataset.samples[idx]
        (forget_idx if y in fset else retain_train_idx).append(idx)

    retain_test_idx, forget_test_idx = [], []
    for idx in test_idx:
        _, y = dataset.samples[idx]
        (forget_test_idx if y in fset else retain_test_idx).append(idx)

    # retain-only VAL split (mirror retain_test_idx logic nhưng trên
    # val_idx) — dùng để best-checkpoint-select trong run_retain_repair mà
    # KHÔNG "nhìn" vào retain_test_idx (tránh leak test set vào lựa chọn
    # checkpoint).
    retain_val_idx, forget_val_idx = [], []
    for idx in val_idx:
        _, y = dataset.samples[idx]
        (forget_val_idx if y in fset else retain_val_idx).append(idx)

    return dict(
        train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
        forget_idx=forget_idx, retain_train_idx=retain_train_idx,
        retain_test_idx=retain_test_idx, forget_test_idx=forget_test_idx,
        retain_val_idx=retain_val_idx, forget_val_idx=forget_val_idx,
        forget_ids=sorted(forget_ids),
        unseen_idx=unseen_idx,
        unseen_classes=sorted(unseen_classes),
    )


In [ ]:
# ===========================================================================
# ▓  CELL D: MODEL ARCHITECTURE — Simple Neck
# ===========================================================================

class ArcMarginProduct(nn.Module):
    """ArcFace margin loss head. Returns cosine-scaled logits."""
    def __init__(self, in_features, out_features, s=32.0, m=0.30):
        super().__init__()
        self.s, self.m = s, m
        self.weight = nn.Parameter(torch.FloatTensor(out_features, in_features))
        nn.init.xavier_uniform_(self.weight)
        self.cos_m = math.cos(m); self.sin_m = math.sin(m)
        self.th    = math.cos(math.pi - m)
        self.mm    = math.sin(math.pi - m) * m

    def forward(self, emb, labels=None):
        emb = F.normalize(emb, dim=1)
        W   = F.normalize(self.weight, dim=1)
        cos = F.linear(emb, W)            # [B, C]
        if labels is None:
            return cos * self.s
        sin  = torch.sqrt((1.0 - cos**2).clamp(1e-7))
        phi  = cos * self.cos_m - sin * self.sin_m
        phi  = torch.where(cos > self.th, phi, cos - self.mm)
        oh   = torch.zeros_like(cos)
        oh.scatter_(1, labels.view(-1,1), 1.0)
        return (oh * phi + (1.0 - oh) * cos) * self.s


class FaceModel(nn.Module):
    """ResNet-18 + Simple Linear Neck (no BN) + ArcFace.
    forward(x, labels) → (embedding [B, D], logits [B, C])"""
    def __init__(self, num_classes: int, embedding_dim: int = 128,
                 pretrained: bool = True, s: float = 32.0, m: float = 0.30):
        super().__init__()
        bb = models.resnet18(
            weights=models.ResNet18_Weights.DEFAULT if pretrained else None
        )
        in_feat = bb.fc.in_features    # 512 for ResNet-18
        bb.fc   = nn.Identity()
        self.backbone      = bb
        self.embedding_dim = embedding_dim

        # ★ Simple neck — single Linear, no BN
        self.neck = nn.Linear(in_feat, embedding_dim, bias=False)

        self.arcface = ArcMarginProduct(embedding_dim, num_classes, s=s, m=m)

    def forward(self, x, labels=None):
        feat   = self.backbone(x)                # [B, 512]
        emb    = self.neck(feat)                  # [B, D]
        emb    = F.normalize(emb, dim=1)         # unit sphere
        logits = self.arcface(emb, labels)
        return emb, logits


def clone_model(model: nn.Module) -> nn.Module:
    return copy.deepcopy(model)


In [ ]:
# ===========================================================================
# ▓  CELL E: ORIGINAL MODEL TRAINING
# ===========================================================================

def train_original_model(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    cfg: Config,
) -> nn.Module:
    """Warmup tuyến tính, sau đó giảm LR thủ công dựa trên CÙNG MỘT best_val
    dùng cho early-stopping.

    v7.3 FIX — BUG PHÁT HIỆN TỪ LOG THỰC TẾ (VGGFace2):
    ReduceLROnPlateau tự theo dõi "best" nội bộ RIÊNG, tách biệt khỏi biến
    best_val ở đây. Vì plateau_scheduler.step() chỉ bắt đầu được gọi từ sau
    warmup (epoch >= warmup_epochs), nếu đỉnh val_acc thật đã đạt được
    TRONG lúc warmup, thì giá trị val_acc thấp đầu tiên sau warmup trở
    thành "best" nội bộ của scheduler — không bao giờ đếm đủ patience để
    giảm LR, trong khi early-stop (so với đỉnh thật) đúng là không cải
    thiện nên đếm đủ patience và dừng training trước khi LR kịp giảm lần
    nào. Fix: dùng chung DUY NHẤT một best_val cho cả early-stop và giảm LR
    (không dùng ReduceLROnPlateau nữa)."""
    device = cfg.device
    model.to(device)
    optimizer = torch.optim.SGD(
        model.parameters(), lr=cfg.original_lr,
        momentum=cfg.original_momentum, weight_decay=cfg.original_wd,
        nesterov=True,
    )
    min_lr = cfg.original_lr * 1e-3
    ce = nn.CrossEntropyLoss()
    best_val, best_state = -1.0, None
    epochs_no_improve = 0

    for epoch in range(cfg.original_epochs):
        if epoch < cfg.warmup_epochs:
            warmup_lr = cfg.original_lr * (epoch + 1) / cfg.warmup_epochs
            for g in optimizer.param_groups:
                g['lr'] = warmup_lr

        model.train(); total_loss = 0.0
        for x, y, _ in train_loader:
            x, y = x.to(device), y.to(device)
            _, logits = model(x, y)
            loss = ce(logits, y)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            total_loss += float(loss)

        val_acc = evaluate_accuracy(model, val_loader, device)

        if val_acc > best_val:
            best_val = val_acc; best_state = copy.deepcopy(model.state_dict())
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if (epoch >= cfg.warmup_epochs
                    and epochs_no_improve % cfg.lr_reduce_patience == 0):
                for g in optimizer.param_groups:
                    new_lr = max(g['lr'] * 0.5, min_lr)
                    g['lr'] = new_lr
                print(f"[Train]   ↓ LR giảm còn {new_lr:.2e} "
                      f"(no_improve={epochs_no_improve}, best_val={best_val:.4f})")

        cur_lr = optimizer.param_groups[0]['lr']
        if (epoch + 1) % 5 == 0 or epoch == cfg.original_epochs - 1:
            print(f"[Train][{epoch+1:02d}/{cfg.original_epochs}]  "
                  f"loss={total_loss/len(train_loader):.4f}  val_acc={val_acc:.4f}  "
                  f"best_val={best_val:.4f}  no_improve={epochs_no_improve}  lr={cur_lr:.2e}")
        if epochs_no_improve >= cfg.early_stop_patience:
            print(f"[Train] Early stop tại epoch {epoch+1}: "
                  f"val_acc không cải thiện sau {cfg.early_stop_patience} epoch "
                  f"(best_val={best_val:.4f}, lr={cur_lr:.2e})")
            break
    model.load_state_dict(best_state)
    return model


In [ ]:
# ===========================================================================
# ▓  CELL F: PROTOTYPE & K-NN UTILITIES
# ===========================================================================
#
# v_C_vgg — PORT nguyên vẹn từ MUFAC-C. Các hàm "group"-based (identity thật)
# TỰ ĐỘNG hoạt động đúng cho VGGFace2 nhờ get_identity_group_ids(): dataset
# VGGFace2 (ImageFolder) KHÔNG có thuộc tính .households, nên hàm rơi vào
# nhánh else (group = classifier label y) — với VGGFace2, "1 group = 1
# identity = 1 class ArcFace" nên nhánh else này CHÍNH XÁC là điều cần —
# không cần viết thêm code riêng cho VGGFace2 ở đây.

@torch.no_grad()
def compute_class_prototypes(
    model: nn.Module,
    loader: DataLoader,
    num_classes: int,
    device: str,
) -> torch.Tensor:
    """Mean unit-norm embedding per CLASSIFIER class. Dùng cho retain-
    prototype anchor (bảo vệ ranh giới phân loại)."""
    model.eval()
    D = model.embedding_dim
    sums   = torch.zeros(num_classes, D)
    counts = torch.zeros(num_classes)
    for x, y, _ in loader:
        z, _ = model(x.to(device), None)
        z = z.cpu()
        for i, cls in enumerate(y.tolist()):
            sums[cls] += z[i]
            counts[cls] += 1
    nonzero = counts > 0
    sums[nonzero] = F.normalize(sums[nonzero], dim=1)
    return sums


@torch.no_grad()
def compute_forget_prototype(
    model: nn.Module,
    forget_loader: DataLoader,
    device: str,
) -> torch.Tensor:
    """Unit-norm mean embedding of forget-set samples. Returns CPU tensor."""
    model.eval()
    embs = [model(x.to(device), None)[0].cpu() for x, _, _ in forget_loader]
    z = torch.cat(embs)
    return F.normalize(z.mean(0, keepdim=True), dim=1).squeeze(0)


def identity_equals_class(forget_ids: List, num_classes: int) -> bool:
    """True nếu identity (forget_ids) trùng với classifier class (VGGFace2 —
    trường hợp CỦA FILE NÀY). False ở MUFAC (forget_ids là family-id dạng
    str, tách biệt khỏi age-class). Dùng để quyết định có áp dụng cơ chế
    'stash/freeze ArcFace row của lớp bị forget' hay không."""
    return len(forget_ids) > 0 and all(
        isinstance(f, int) and 0 <= f < num_classes for f in forget_ids
    )


def get_identity_group_ids(dataset) -> List:
    """group_id của từng sample theo index gốc, dùng cho K-NN 'local
    identity'. VGGFace2: KHÔNG có .households → rơi vào nhánh else, group =
    nhãn phân loại y = chính identity (đúng ý nghĩa cho dataset này)."""
    if hasattr(dataset, "households"):
        return dataset.households
    return [y for _, y in dataset.samples]


def compute_group_prototypes(
    model: nn.Module,
    loader: DataLoader,          # phải yield (x, y, orig_idx) — từ IndexedSubset
    group_ids: List,             # group_ids[orig_idx] = group label
    device: str,
) -> Dict:
    """Mean unit-norm embedding theo GROUP (identity thật)."""
    model.eval()
    sums: Dict = {}
    counts: Dict = {}
    with torch.no_grad():
        for x, _, idx in loader:
            z, _ = model(x.to(device), None)
            z = z.cpu()
            idx_list = idx.tolist() if torch.is_tensor(idx) else list(idx)
            for i, oidx in enumerate(idx_list):
                g = group_ids[oidx]
                if g not in sums:
                    sums[g] = torch.zeros(model.embedding_dim)
                    counts[g] = 0
                sums[g] += z[i]
                counts[g] += 1
    return {g: F.normalize(sums[g].unsqueeze(0), dim=1).squeeze(0)
            for g in sums if counts[g] > 0}


def find_knn_retain_groups(
    forget_proto: torch.Tensor,
    group_prototypes: Dict,
    forget_group_ids: List,
    k: int,
) -> List:
    """K retain identities gần forget_proto nhất."""
    fp = F.normalize(forget_proto.unsqueeze(0), dim=1)
    fset = set(forget_group_ids)
    items = [(g, p) for g, p in group_prototypes.items() if g not in fset]
    if not items:
        return []
    ids = [g for g, _ in items]
    mat = torch.stack([p for _, p in items])
    sims = (mat @ fp.T).squeeze(1)
    k = min(k, len(ids))
    topk = torch.topk(sims, k=k).indices.tolist()
    return [ids[i] for i in topk]


def build_local_anchor_loader_grouped(
    base_dataset,
    retain_train_idx: List[int],
    group_ids: List,
    anchor_group_ids: List,
    batch_size: int,
    num_workers: int,
) -> Optional[DataLoader]:
    """DataLoader chỉ chứa retain-sample thuộc K identity gần forget nhất."""
    anchor_set = set(anchor_group_ids)
    anchor_indices = [idx for idx in retain_train_idx if group_ids[idx] in anchor_set]
    if not anchor_indices:
        print("[VGGFace2] ⚠ No anchor samples found.")
        return None
    ds = IndexedSubset(base_dataset, anchor_indices)
    print(f"[VGGFace2] Anchor loader: {len(anchor_group_ids)} identities, {len(anchor_indices)} samples")
    return DataLoader(ds, batch_size=batch_size, shuffle=True,
                      num_workers=num_workers, pin_memory=True)


In [ ]:
# ===========================================================================
# ▓  CELL G: EWC UTILITIES
# ===========================================================================

@torch.no_grad()
def snapshot_neck_params(model: nn.Module) -> Dict[str, torch.Tensor]:
    """Save a copy of neck (and layer4) parameter values as θ* for EWC."""
    snap = {}
    for name, param in model.named_parameters():
        if "neck" in name or "backbone.layer4" in name:
            snap[name] = param.data.clone().cpu()
    return snap


def compute_ewc_fisher(
    model: nn.Module,
    retain_loader: DataLoader,
    device: str,
    n_batches: int = 30,
) -> Dict[str, torch.Tensor]:
    """
    Diagonal Fisher Information trên retain set cho neck + layer4 params.
    F_i = (1/N) Σ (∂CE/∂θ_i)² — F_i cao → θ_i quan trọng cho retain → EWC
    phạt nặng nếu θ_i lệch khỏi θ*.
    """
    model.eval()
    fisher: Dict[str, torch.Tensor] = {}
    target_names = set()
    for name, param in model.named_parameters():
        if "neck" in name or "backbone.layer4" in name:
            fisher[name] = torch.zeros_like(param.data, device='cpu')
            target_names.add(name)

    ce = nn.CrossEntropyLoss()
    count = 0
    for x, y, _ in retain_loader:
        if count >= n_batches:
            break
        x, y = x.to(device), y.to(device)
        model.zero_grad()
        _, logits = model(x, y)
        loss = ce(logits, y)
        loss.backward()
        for name, param in model.named_parameters():
            if name in target_names and param.grad is not None:
                fisher[name] += param.grad.data.pow(2).cpu()
        count += 1

    for name in fisher:
        fisher[name] /= max(count, 1)

    total_params = sum(v.numel() for v in fisher.values())
    mean_f = sum(float(v.mean()) for v in fisher.values()) / max(len(fisher), 1)
    print(f"[VGGFace2] EWC Fisher computed: {len(fisher)} param groups, "
          f"{total_params:,} params, mean_F={mean_f:.6f}")
    return fisher


def compute_ewc_loss(
    model: nn.Module,
    theta_star: Dict[str, torch.Tensor],
    fisher: Dict[str, torch.Tensor],
    w_ewc: float,
    device: str,
) -> torch.Tensor:
    """L_ewc = (w_ewc/2) × Σ_i F_i × (θ_i − θ*_i)²"""
    loss = torch.tensor(0.0, device=device, requires_grad=True)
    for name, param in model.named_parameters():
        if name in fisher and name in theta_star:
            F_i      = fisher[name].to(device)
            theta_i  = theta_star[name].to(device)
            loss = loss + (F_i * (param - theta_i).pow(2)).sum()
    return (w_ewc / 2.0) * loss


In [ ]:
# ===========================================================================
# ▓  CELL H: LOSS FUNCTIONS
# ===========================================================================

class ForgetLossV7(nn.Module):
    """Four-component forget loss. repel_margin = -0.30: pair repulsion
    đẩy tới khi cos(z_fi, z_fj) < -0.3 — scatter thật, không chỉ decorrelate.
    L_div (Hướng C) thêm KL-to-unseen — xem ghi chú ở Cell I
    (_sample_unseen_batch_vgg) về cách VGGFace2 khác MUFAC ở bước này."""
    def __init__(self, cfg: Config, num_classes: int):
        super().__init__()
        self.cfg   = cfg
        self.C     = num_classes
        self.log_C = math.log(num_classes)

    def forward(
        self,
        z_f:       torch.Tensor,   # [B, D] unit-norm forget embeddings
        old_proto: torch.Tensor,   # [D]    frozen prototype (on device)
        logits_f:  torch.Tensor,   # [B, C] ArcFace logits
        w_erase:   float = None,   # override for staged schedule
        w_repel:   float = None,
        w_kl_uni:  float = None,   # override for staged Migrate→Shatter schedule
        w_div:     float = None,   # override for staged Migrate→Shatter schedule
        logits_f_unseen_old: torch.Tensor = None,  # [B,C] logits của old_model trên batch unseen
    ) -> Tuple[torch.Tensor, Dict]:
        B = z_f.size(0)
        w_e  = w_erase  if w_erase  is not None else self.cfg.w_erase
        w_r  = w_repel  if w_repel  is not None else self.cfg.w_repel
        w_kl = w_kl_uni if w_kl_uni is not None else self.cfg.w_kl_uniform
        w_d  = w_div    if w_div    is not None else self.cfg.w_div

        # ── Anti-prototype push ────────────────────────────────────────────
        L_erase = (z_f * old_proto.unsqueeze(0)).sum(dim=1).mean()

        # ── KL-to-Uniform (calibration) ───────────────────────────────────
        p        = F.softmax(logits_f, dim=1)
        H        = -(p * torch.log(p + 1e-8)).sum(dim=1).mean()
        L_kl_uni = self.log_C - H

        # ── Pair repulsion with aggressive margin ─────────────────────────
        if B > 1:
            sim_mat = z_f @ z_f.T                  # [B, B]
            mask    = ~torch.eye(B, dtype=torch.bool, device=z_f.device)
            L_repel = F.relu(sim_mat[mask] - self.cfg.repel_margin).mean()
        else:
            L_repel = torch.zeros(1, device=z_f.device).squeeze()

        # ── Hướng C: KL-to-unseen distillation ─────────────────────────────
        # KL(P_orig(x_unseen) ‖ P_forget(x_forget)), cùng cách dùng F.kl_div
        # như RetainLossV7.forward (KD term ở dưới): target = p_orig_unseen
        # (no-grad, từ old_model), input = log_softmax(logits_f) (CÓ grad) —
        # minimize kéo p_forget → p_orig_unseen.
        if logits_f_unseen_old is not None and w_d > 0:
            log_p_f       = F.log_softmax(logits_f, dim=1)
            p_orig_unseen = F.softmax(logits_f_unseen_old, dim=1)
            L_div = F.kl_div(log_p_f, p_orig_unseen, reduction='batchmean')
        else:
            L_div = torch.zeros(1, device=z_f.device).squeeze()

        total = (w_e * L_erase
               + w_kl * L_kl_uni
               + w_r  * L_repel
               + w_d  * L_div)

        parts = {
            "L_erase":      float(L_erase.detach()),
            "L_kl_uni":     float(L_kl_uni.detach()),
            "L_repel":      float(L_repel.detach()),
            "L_div":        float(L_div.detach()),
            "entropy":      float(H.detach()),
            "cos_to_proto": float(L_erase.detach()),
        }
        return total, parts


class RetainLossV7(nn.Module):
    """Retain loss (KD + K-NN local anchor + CE) + retain-prototype anchor.

    L_proto_retain là đối xứng trực tiếp với L_erase của ForgetLossV7:
      L_erase        = mean  cos(z_f, p_forget_old)   → ĐẨY forget ra xa prototype
      L_proto_retain = mean(1 - cos(z_r, p_{y_r}))    → KÉO retain VỀ prototype của
                                                          đúng lớp của nó
    """
    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        self.T   = cfg.kd_temperature
        self.ce  = nn.CrossEntropyLoss()

    def forward(
        self,
        logits_r:        torch.Tensor,
        logits_r_old:    torch.Tensor,
        y_r:             torch.Tensor,
        z_n_new:         Optional[torch.Tensor] = None,
        z_n_old:         Optional[torch.Tensor] = None,
        w_local:         float = None,   # override for staged schedule
        z_r_new:         Optional[torch.Tensor] = None,   # [B, D] retain embeddings
        retain_prototypes: Optional[torch.Tensor] = None, # [C, D] frozen, trên device
        w_proto:         float = None,   # override for staged schedule
    ) -> Tuple[torch.Tensor, Dict]:
        w_l = w_local if w_local is not None else self.cfg.w_kd_local
        w_p = w_proto if w_proto is not None else self.cfg.w_proto_retain

        # Global KL distillation
        log_p = F.log_softmax(logits_r     / self.T, dim=1)
        p_old = F.softmax(   logits_r_old  / self.T, dim=1)
        kl    = (self.T ** 2) * F.kl_div(log_p, p_old, reduction='batchmean')

        # CE
        ce = self.ce(logits_r, y_r)

        # Local K-NN cosine anchor (gated by w_l — zero in Phase 1)
        if z_n_new is not None and z_n_old is not None and w_l > 0:
            cos_sim  = (z_n_new * z_n_old).sum(dim=1)
            L_anchor = (1.0 - cos_sim).mean()
        else:
            L_anchor = torch.zeros(1, device=logits_r.device).squeeze()

        # Retain-prototype anchor (luôn bật, kể cả Phase 1)
        if (self.cfg.use_proto_retain and z_r_new is not None
                and retain_prototypes is not None and w_p > 0):
            proto_y  = retain_prototypes[y_r]            # [B, D]
            cos_p    = (z_r_new * proto_y).sum(dim=1)
            L_proto  = (1.0 - cos_p).mean()
        else:
            L_proto = torch.zeros(1, device=logits_r.device).squeeze()

        total = (self.cfg.w_kd_global * kl
               + w_l                  * L_anchor
               + self.cfg.w_ce_retain * ce
               + w_p                  * L_proto)

        parts = {
            "L_kd":     float(kl.detach()),
            "L_anchor": float(L_anchor.detach()) if isinstance(L_anchor, torch.Tensor) else 0.0,
            "L_ce":     float(ce.detach()),
            "L_proto":  float(L_proto.detach()) if isinstance(L_proto, torch.Tensor) else 0.0,
        }
        return total, parts


In [ ]:
# ===========================================================================
# ▓  CELL H2: PCGrad + GRADIENT PROJECTION UTILITY
# ===========================================================================

def pcgrad_project(g_f: torch.Tensor, g_r: torch.Tensor) -> torch.Tensor:
    """
    Project g_f onto orthogonal complement of g_r when conflicting.
    g_f' = g_f − (⟨g_f,g_r⟩/‖g_r‖²)·g_r  if ⟨g_f,g_r⟩ < 0, else g_f.
    """
    inner = torch.dot(g_f, g_r)
    if inner >= 0:
        return g_f
    g_r_sq = torch.dot(g_r, g_r)
    if g_r_sq < 1e-12:
        return g_f
    return g_f - (inner / g_r_sq) * g_r


def build_orthonormal_basis(proto_matrix: Optional[torch.Tensor]) -> Optional[torch.Tensor]:
    """v7.5 — proto_matrix: [K, D] (K local retain-prototype véc-tơ, không
    nhất thiết trực giao với nhau). Trả về basis trực giao hoá [K', D] (hàng
    trực giao đơn vị, K' <= K) trương ra CÙNG không gian con — dùng QR trên
    ma trận chuyển vị. Trả None nếu proto_matrix rỗng/None (tắt gradient
    projection một cách an toàn — fallback về hành vi cũ)."""
    if proto_matrix is None or proto_matrix.numel() == 0:
        return None
    Q, _ = torch.linalg.qr(proto_matrix.T, mode='reduced')   # Q: [D, K']
    return Q.T.contiguous()                                   # [K', D]


def project_out_prototype_directions(
    grad:        torch.Tensor,   # [D, H] — gradient của neck.weight
    proto_basis: torch.Tensor,   # [K', D] trực giao đơn vị, cùng device với grad
) -> torch.Tensor:
    """v7.5 — GRADIENT PROJECTION (Local Prototype Erasure theo đúng nghĩa
    đen của tên phương pháp).

    neck.weight ánh xạ trực tiếp feat → embedding: emb = W @ feat. Với BẤT KỲ
    feat nào, thay đổi embedding do một update ΔW gây ra là ΔW @ feat ∈ R^D.
    Ta muốn thành phần của thay đổi đó dọc theo mỗi local retain-prototype
    p_k LUÔN BẰNG 0 — tức p_k^T @ ΔW = 0 (một vector H-chiều) với mọi k. Đây
    là một ràng buộc CỨNG ở cấp tham số (không phải cấp loss), nên không bị
    'đánh đổi ngược' qua các bước train sau như cách các loss-term vẫn bị
    (khác PCGrad — vốn chỉ giải quyết xung đột theo batch retain HIỆN TẠI,
    nhiễu theo batch và không cố định qua thời gian).

    Cài đặt: coi mỗi CỘT của grad (∈ R^D, D=embedding_dim) là một vector cần
    chiếu lên phần bù trực giao của span(proto_basis):
        grad_perp = grad − proto_basis^T @ (proto_basis @ grad)
    """
    proj = proto_basis.T @ (proto_basis @ grad)   # [D, H]
    return grad - proj


In [ ]:
# ===========================================================================
# ▓  CELL I: STAGED UNLEARNING LOOP — LPEU (Hướng C)
# ===========================================================================

def get_stage_weights(epoch: int, cfg: Config) -> Tuple[float, float, float, float]:
    """
    Phase 1 — ERASE (epoch < stage1_epochs): Anchor OFF. Max forgetting.
    Phase 2 — REFINE (stage1 ≤ epoch < stage1+stage2): Anchor 50%.
    Phase 3 — STABLE (epoch ≥ stage1+stage2): Anchor 100%. Light forgetting.

    Hàm này chỉ còn quyết định w_local (anchor) và boost. w_erase/w_repel
    trả về ở đây KHÔNG còn được dùng trực tiếp (giữ lại trong return để
    không phá vỡ signature) — 2 giá trị đó nay do get_forget_subphase_weights()
    quyết định, theo lịch Migrate→Shatter tách riêng khỏi lịch anchor.
    """
    e1 = cfg.stage1_epochs
    e2 = cfg.stage1_epochs + cfg.stage2_epochs

    if epoch < e1:
        return 0.0, cfg.w_erase, cfg.w_repel, cfg.pcgrad_boost
    elif epoch < e2:
        return cfg.w_kd_local * 0.5, cfg.w_erase * 0.75, cfg.w_repel * 0.67, cfg.pcgrad_boost
    else:
        return cfg.w_kd_local, cfg.w_erase * 0.5, cfg.w_repel * 0.33, cfg.pcgrad_boost * 0.5


def get_forget_subphase_weights(
    epoch: int,
    proto_drop_so_far: float,
    initial_proto_sim: float,
    in_shatter: bool,
    cfg: Config,
) -> Tuple[float, float, float, float, bool]:
    """
    Migrate → Shatter: tách lịch w_erase / w_repel khỏi lịch anchor
    (get_stage_weights ở trên). 2 sub-phase NỐI TIẾP, 1 chiều:

      MIGRATE  (in_shatter=False): w_erase=cfg.w_erase đầy đủ, w_repel=0.
               Dồn hết lực "di chuyển" cụm forget ra khỏi prototype cũ,
               không bị L_repel giành lực / lấp lại vùng vừa dọn.
      SHATTER  (in_shatter=True):  w_erase hạ còn một sàn dư
               (shatter_erase_floor × cfg.w_erase, chống trôi ngược),
               w_repel=cfg.w_repel đầy đủ — giờ mới "phá rã" cụm đã dịch.

    Điều kiện chuyển MIGRATE → SHATTER (đánh giá 1 lần mỗi epoch, dùng
    proto_drop đo được ở CUỐI epoch trước):
      - proto_drop_so_far / (1 + initial_proto_sim) ≥ migrate_target_fraction,
        HOẶC
      - epoch ≥ migrate_max_epochs (an toàn).
    Một khi đã sang SHATTER thì KHÔNG quay lại MIGRATE.

    w_kl_uni và w_div (Hướng C) đi CÙNG lịch Migrate→Shatter: 0 trong
    MIGRATE, đủ trọng số trong SHATTER — tránh giành lực với L_erase ngay
    từ đầu MIGRATE.
    """
    max_possible_drop = 1.0 + initial_proto_sim
    achieved_fraction = (
        proto_drop_so_far / max_possible_drop if max_possible_drop > 1e-8 else 1.0
    )

    if not in_shatter:
        crossed_threshold = achieved_fraction >= cfg.migrate_target_fraction
        crossed_epoch_cap = epoch >= cfg.migrate_max_epochs
        if crossed_threshold or crossed_epoch_cap:
            in_shatter = True

    if not in_shatter:
        w_erase  = cfg.w_erase
        w_repel  = 0.0
        w_kl_uni = 0.0
        w_div    = 0.0
    else:
        w_erase  = cfg.w_erase * cfg.shatter_erase_floor
        w_repel  = cfg.w_repel
        w_kl_uni = cfg.w_kl_uniform
        w_div    = cfg.w_div

    return w_erase, w_repel, w_kl_uni, w_div, in_shatter


def _sample_unseen_batch_vgg(
    batch_size: int,
    unseen_pool_images: torch.Tensor,
    device,
) -> torch.Tensor:
    """v_C_vgg — Hướng C cho VGGFace2. Ở MUFAC, việc "khớp lớp" (age-class)
    có ý nghĩa vì age-class là nhãn THÔ dùng chung giữa forget và unseen
    (xem _sample_matched_unseen_batch trong lpeu-v7-mufac-person-C.ipynb).
    Ở VGGFace2, identity CHÍNH LÀ class, và các identity trong D_unseen
    TÁCH BIỆT HOÀN TOÀN (disjoint) với identity forget — KHÔNG có nhãn
    chung nào để match. Thay vào đó, lấy NGẪU NHIÊN (có hoàn lại) một batch
    cùng kích thước từ TOÀN BỘ D_unseen (không phân biệt danh tính) làm
    target cho KL — vẫn giữ đúng Ý NGHĨA cốt lõi của L_div: kéo hành vi
    model-forget trên ảnh forget về đúng hành vi model GỐC trên dữ liệu
    THẬT SỰ chưa từng thấy, chỉ khác là không thể class-matched."""
    n = unseen_pool_images.size(0)
    idx = torch.randint(0, n, (batch_size,))
    return unseen_pool_images[idx].to(device)


def run_lpeu_v7_unlearning(
    model:              nn.Module,
    old_model:          nn.Module,
    forget_loader:      DataLoader,
    retain_loader:      DataLoader,
    retain_test_loader: DataLoader,
    anchor_loader:      Optional[DataLoader],
    forget_ids:         List,
    num_classes:        int,
    cfg:                Config,
    retain_acc_ref:     float,
    fisher:             Optional[Dict[str, torch.Tensor]] = None,
    theta_star:         Optional[Dict[str, torch.Tensor]] = None,
    retain_prototypes:  Optional[torch.Tensor] = None,   # [C, D] frozen, CPU
    local_retain_prototypes: Optional[torch.Tensor] = None,  # [K, D] frozen, CPU — gradient projection
    unseen_pool_images: Optional[torch.Tensor] = None,   # v_C_vgg — [N,C,H,W] CPU, cho L_div (xem _sample_unseen_batch_vgg)
) -> nn.Module:
    """
    LPEU staged unlearning loop.

    Per-iteration:
    ═══════════════════════════════════════════════════════════════
    [A] RETAIN STEP:
      1. Forward retain batch → logits_r
      2. Forward anchor batch → z_n_new (Phase 2+3 only)
      3. L_retain = KL + staged_anchor + CE  [+ EWC if use_ewc]
      4. backward → save neck grads for PCGrad → step → STASH ArcFace

    [B] FORGET STEP (PCGrad + gradient-projection protected):
      1. Forward forget batch → (z_f, logits_f)
      2. L_forget = staged_erase + kl_uni + staged_repel + staged_div
      3. backward → PCGrad project neck grads → boost → step
      4. RESTORE ArcFace retain weights (VGGFace2: chỉ các hàng RETAIN —
         xem identity_equals_class bên dưới)
    ═══════════════════════════════════════════════════════════════
    """
    device = cfg.device
    model.to(device)
    old_model.to(device)
    old_model.eval()
    for p in old_model.parameters():
        p.requires_grad = False

    # ── Trainable parameters ──────────────────────────────────────────────
    for p in model.parameters():
        p.requires_grad = False

    for name, p in model.named_parameters():
        if "neck" in name or "arcface" in name:
            p.requires_grad = True

    layer4_params = []
    for name, p in model.named_parameters():
        if "backbone.layer4" in name:
            p.requires_grad = True
            layer4_params.append(p)

    neck_arcface_params = [p for n, p in model.named_parameters()
                           if p.requires_grad and "backbone.layer4" not in n]
    all_trainable = [p for p in model.parameters() if p.requires_grad]

    pcgrad_param_names = set(n for n, p in model.named_parameters()
                             if p.requires_grad and ("neck" in n or "backbone.layer4" in n))

    n_train = sum(p.numel() for p in all_trainable)
    print(f"[VGGFace2] Trainable: {n_train:,} params  |  "
          f"EWC: {'ON' if cfg.use_ewc and fisher else 'OFF'}")

    # ── Optimizer ─────────────────────────────────────────────────────────
    optimizer = torch.optim.AdamW(
        [
            {"params": neck_arcface_params, "lr": cfg.unlearn_lr},
            {"params": layer4_params,       "lr": cfg.unlearn_lr * cfg.layer4_lr_mult},
        ],
        weight_decay=cfg.unlearn_wd,
        betas=(0.9, 0.999),
    )
    total_epochs = cfg.unlearn_epochs
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=total_epochs, eta_min=cfg.unlearn_lr * 0.05,
    )

    forget_crit = ForgetLossV7(cfg, num_classes)
    retain_crit = RetainLossV7(cfg)

    forget_proto = compute_forget_prototype(old_model, forget_loader, device)
    forget_proto = forget_proto.to(device)
    _p_cpu       = forget_proto.cpu()
    initial_proto_sim = float(
        compute_forget_prototype(old_model, forget_loader, device).dot(_p_cpu)
    )

    # Checkpoint selection: mốc fgt_acc BAN ĐẦU (của old_model, trước khi
    # unlearn) để đo % tiến gần tới rc (1/num_classes) mỗi epoch — dùng làm
    # tín hiệu forget bổ sung bên cạnh proto_drop thuần representation-level.
    initial_fgt_acc = evaluate_accuracy(old_model, forget_loader, device)
    rc = 1.0 / num_classes

    # Retain-prototype anchor: frozen [C, D] prototypes tính TRƯỚC unlearning
    retain_proto_dev = (
        retain_prototypes.to(device)
        if (cfg.use_proto_retain and retain_prototypes is not None) else None
    )

    # Gradient projection setup: basis trực giao từ local KNN retain
    # prototypes (K identity retain gần forget-identity nhất). Chỉ áp dụng
    # lên neck.weight.
    proto_basis_dev = None
    if cfg.use_grad_projection and local_retain_prototypes is not None:
        proto_basis_dev = build_orthonormal_basis(local_retain_prototypes)
        if proto_basis_dev is not None:
            proto_basis_dev = proto_basis_dev.to(device)
            print(f"[VGGFace2] Gradient projection ON — basis rank={proto_basis_dev.shape[0]} "
                  f"(từ {local_retain_prototypes.shape[0]} local prototypes)")

    # VGGFace2: identity CHÍNH LÀ classifier class → identity_equals_class
    # trả True → chỉ các hàng ArcFace RETAIN (không thuộc forget_ids) bị
    # stash/restore mỗi forget-step; các hàng FORGET được TỰ DO thay đổi
    # (đây là điểm khác biệt cấu trúc so với MUFAC — xem Cell B).
    if identity_equals_class(forget_ids, num_classes):
        fset = set(forget_ids)
        retain_cls_t = torch.tensor(
            [c for c in range(num_classes) if c not in fset], dtype=torch.long
        )
    else:
        retain_cls_t = torch.arange(num_classes, dtype=torch.long)

    best_state   = copy.deepcopy(model.state_dict())
    best_balance = -1.0
    violation_streak = 0

    print(f"\n[VGGFace2] Starting staged unlearning — {total_epochs} epochs total")
    print(f"     Anchor schedule   — Phase 1 (ERASE):  epoch 0–{cfg.stage1_epochs-1}      anchor=OFF")
    print(f"     Anchor schedule   — Phase 2 (REFINE): epoch {cfg.stage1_epochs}–{cfg.stage1_epochs+cfg.stage2_epochs-1}  anchor=50%")
    print(f"     Anchor schedule   — Phase 3 (STABLE): epoch {cfg.stage1_epochs+cfg.stage2_epochs}–{total_epochs-1}  anchor=100%")
    print(f"     Forget schedule   — MIGRATE (w_erase={cfg.w_erase}, w_repel=0, w_div=0) → "
          f"SHATTER (w_erase={cfg.w_erase*cfg.shatter_erase_floor:.2f}, w_repel={cfg.w_repel}, w_div={cfg.w_div}), "
          f"switch @ proto_drop≥{cfg.migrate_target_fraction:.0%} of max or epoch≥{cfg.migrate_max_epochs}")
    print(f"     Hướng C: L_div (KL-to-unseen, KHÔNG class-matched — xem _sample_unseen_batch_vgg) — "
          f"{'unseen_pool_images đã truyền, sẽ tính L_div trong SHATTER' if unseen_pool_images is not None else 'KHÔNG truyền unseen_pool_images — L_div sẽ luôn=0 (Hướng C tắt)'}")
    print(f"     Initial proto_sim = {initial_proto_sim:.4f}")

    anchor_iter = iter(anchor_loader) if anchor_loader else None
    in_shatter      = False   # Migrate→Shatter state — 1 chiều, False→True
    proto_drop_prev = 0.0     # proto_drop đo cuối epoch TRƯỚC (epoch 0 dùng 0.0 → luôn MIGRATE)

    for epoch in range(total_epochs):
        model.train()

        w_local, _, _, boost = get_stage_weights(epoch, cfg)
        w_erase, w_repel, w_kl_uni, w_div, in_shatter = get_forget_subphase_weights(
            epoch, proto_drop_prev, initial_proto_sim, in_shatter, cfg
        )
        phase_name = (
            "ERASE" if epoch < cfg.stage1_epochs else
            "REFINE" if epoch < cfg.stage1_epochs + cfg.stage2_epochs else
            "STABLE"
        )
        subphase_name = "SHATTER" if in_shatter else "MIGRATE"

        S = {k: 0. for k in ["kd","anchor","ce","proto","erase","kl_uni","repel","div","entropy","ewc","n_conf","steps"]}
        forget_iter = iter(forget_loader)

        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)

            # ══════════════════════════════════════════════════════════════
            # [A] RETAIN STEP
            # ══════════════════════════════════════════════════════════════
            z_r, logits_r = model(x_r, y_r)
            with torch.no_grad():
                _, logits_r_old = old_model(x_r, y_r)

            z_n_new = z_n_old = None
            if anchor_iter is not None and w_local > 0:
                try:
                    x_n, _, _ = next(anchor_iter)
                except StopIteration:
                    anchor_iter = iter(anchor_loader)
                    x_n, _, _ = next(anchor_iter)
                x_n = x_n.to(device)
                z_n_new, _ = model(x_n, None)
                with torch.no_grad():
                    z_n_old, _ = old_model(x_n, None)

            loss_r, parts_r = retain_crit(
                logits_r, logits_r_old, y_r, z_n_new, z_n_old,
                w_local=w_local,
                z_r_new=z_r, retain_prototypes=retain_proto_dev,
            )

            ewc_val = 0.0
            if cfg.use_ewc and fisher is not None and theta_star is not None:
                L_ewc  = compute_ewc_loss(model, theta_star, fisher, cfg.w_ewc, device)
                loss_r = loss_r + L_ewc
                ewc_val = float(L_ewc.detach())

            optimizer.zero_grad()
            loss_r.backward()
            torch.nn.utils.clip_grad_norm_(all_trainable, cfg.grad_clip_retain)

            retain_grads: Dict[str, torch.Tensor] = {}
            for name, param in model.named_parameters():
                if name in pcgrad_param_names and param.grad is not None:
                    retain_grads[name] = param.grad.detach().clone()

            optimizer.step()

            stashed = model.arcface.weight.data[retain_cls_t].clone().cpu()

            S["kd"]     += parts_r["L_kd"]
            S["anchor"] += parts_r["L_anchor"]
            S["ce"]     += parts_r["L_ce"]
            S["proto"]  += parts_r["L_proto"]
            S["ewc"]    += ewc_val

            # ══════════════════════════════════════════════════════════════
            # [B] FORGET STEP — PCGrad protected
            # ══════════════════════════════════════════════════════════════
            try:
                x_f, _, _ = next(forget_iter)
            except StopIteration:
                forget_iter = iter(forget_loader)
                x_f, _, _ = next(forget_iter)
            x_f = x_f.to(device)

            # Hướng C: batch unseen (KHÔNG class-matched — xem
            # _sample_unseen_batch_vgg) để tính L_div. Chỉ tính khi w_div>0
            # (SHATTER) — tránh forward thừa trong MIGRATE.
            logits_f_unseen_old = None
            if unseen_pool_images is not None and w_div > 0:
                x_u = _sample_unseen_batch_vgg(x_f.size(0), unseen_pool_images, device)
                with torch.no_grad():
                    _, logits_f_unseen_old = old_model(x_u, None)

            z_f, logits_f = model(x_f, None)
            loss_f, parts_f = forget_crit(
                z_f, forget_proto, logits_f,
                w_erase=w_erase,
                w_repel=w_repel,
                w_kl_uni=w_kl_uni,
                w_div=w_div,
                logits_f_unseen_old=logits_f_unseen_old,
            )

            optimizer.zero_grad()
            loss_f.backward()
            torch.nn.utils.clip_grad_norm_(all_trainable, cfg.grad_clip_forget)

            n_conflicts = 0
            manual_grads: Dict[str, torch.Tensor] = {}
            for name, param in model.named_parameters():
                if name not in pcgrad_param_names or param.grad is None:
                    continue
                g_f_flat = param.grad.detach().flatten()
                if name in retain_grads:
                    g_r_flat = retain_grads[name].flatten()
                    if torch.dot(g_f_flat, g_r_flat) < 0:
                        n_conflicts += 1
                    g_f_proj = pcgrad_project(g_f_flat, g_r_flat)
                else:
                    g_f_proj = g_f_flat
                manual_grads[name] = (g_f_proj * boost).reshape(param.shape)
                param.grad = None

            if proto_basis_dev is not None and "neck.weight" in manual_grads:
                manual_grads["neck.weight"] = project_out_prototype_directions(
                    manual_grads["neck.weight"], proto_basis_dev
                )

            optimizer.step()

            with torch.no_grad():
                for name, param in model.named_parameters():
                    if name in manual_grads:
                        lr_mult = cfg.layer4_lr_mult if "backbone.layer4" in name else 1.0
                        param -= cfg.forget_manual_lr * lr_mult * manual_grads[name]

            model.arcface.weight.data[retain_cls_t] = stashed.to(device)

            S["erase"]   += parts_f["L_erase"]
            S["kl_uni"]  += parts_f["L_kl_uni"]
            S["repel"]   += parts_f["L_repel"]
            S["div"]     += parts_f["L_div"]
            S["entropy"] += parts_f["entropy"]
            S["n_conf"]  += n_conflicts
            S["steps"]   += 1

            # ══════════════════════════════════════════════════════════════
            # [C] IN-LOOP RETAIN REPAIR
            # ══════════════════════════════════════════════════════════════
            if cfg.use_inloop_repair:
                _, logits_r_rep = model(x_r, y_r)
                with torch.no_grad():
                    _, logits_r_old_rep = old_model(x_r, y_r)
                log_p_rep = F.log_softmax(logits_r_rep / cfg.kd_temperature, dim=1)
                p_old_rep = F.softmax(logits_r_old_rep / cfg.kd_temperature, dim=1)
                loss_rep  = cfg.w_repair * (cfg.kd_temperature ** 2) * \
                            F.kl_div(log_p_rep, p_old_rep, reduction='batchmean')
                optimizer.zero_grad()
                loss_rep.backward()
                torch.nn.utils.clip_grad_norm_(all_trainable, cfg.grad_clip_retain)
                optimizer.step()
                model.arcface.weight.data[retain_cls_t] = stashed.to(device)

        scheduler.step()

        # ── Per-epoch eval ────────────────────────────────────────────────
        n          = max(1, S["steps"])
        ret_acc    = evaluate_accuracy(model, retain_test_loader, device)
        fgt_acc    = evaluate_accuracy(model, forget_loader, device)
        cur_proto  = compute_forget_prototype(model, forget_loader, device)  # CPU
        proto_sim  = float(cur_proto.dot(_p_cpu))
        proto_drop = initial_proto_sim - proto_sim
        proto_drop_prev = proto_drop

        print(
            f"[VGGFace2][{epoch+1:02d}/{total_epochs}][{phase_name}/{subphase_name}]  "
            f"erase={S['erase']/n:.4f}  repel={S['repel']/n:.4f}  kl_uni={S['kl_uni']/n:.4f}  "
            f"div={S['div']/n:.4f}  "
            f"w_erase={w_erase:.3f}  w_repel={w_repel:.3f}  w_kl_uni={w_kl_uni:.3f}  w_div={w_div:.3f}  "
            f"anc={S['anchor']/n:.4f}  proto_r={S['proto']/n:.4f}  ewc={S['ewc']/n:.4f}  "
            f"proto_sim={proto_sim:.4f}(drop:{proto_drop:+.4f})  "
            f"H={S['entropy']/n:.2f}/{math.log(num_classes):.2f}  "
            f"fgt_acc={fgt_acc:.4f}  ret_acc={ret_acc:.4f}  "
            f"conf={S['n_conf']/n:.1f}  boost={boost:.0f}×  "
            f"anchor_w={w_local:.1f}"
        )

        # ── Safety guardrail ────────────────────────────────────────────
        floor = cfg.retain_acc_floor * retain_acc_ref
        if epoch >= cfg.safety_grace_epochs and ret_acc < floor:
            violation_streak += 1
            print(f"[VGGFace2] ⚠ retain_acc dưới floor ({ret_acc:.4f} < {floor:.4f})  "
                  f"streak={violation_streak}/{cfg.safety_patience}")
            if violation_streak >= cfg.safety_patience:
                print(f"[VGGFace2] ⚠ SAFETY STOP epoch {epoch+1}: vi phạm floor "
                      f"{cfg.safety_patience} epoch liên tiếp")
                model.load_state_dict(best_state)
                break
        else:
            violation_streak = 0

        # Best checkpoint: combined proto-drop + accuracy-drop forget score,
        # weighted against retain, gated at checkpoint_retain_gate.
        retain_score = ret_acc / retain_acc_ref
        proto_forget_score = max(0.0, proto_drop / (1.0 + initial_proto_sim))
        acc_forget_span  = max(initial_fgt_acc - rc, 1e-6)
        acc_forget_score = min(1.0, max(0.0, (initial_fgt_acc - fgt_acc) / acc_forget_span))
        combined_forget_score = 0.5 * proto_forget_score + 0.5 * acc_forget_score
        gate = cfg.checkpoint_retain_gate
        if retain_score >= gate:
            balance = retain_score * (0.7 + 0.3 * combined_forget_score)
        else:
            balance = max(combined_forget_score, 1e-3) * (retain_score / gate)

        if balance > best_balance:
            best_balance = balance
            best_state   = copy.deepcopy(model.state_dict())
            print(f"          ★ New best  balance={best_balance:.4f}  "
                  f"proto_drop={proto_drop:+.4f}  proto_fs={proto_forget_score:.3f}  "
                  f"acc_fs={acc_forget_score:.3f} (fgt_acc={fgt_acc:.4f}→rc={rc:.4f})  "
                  f"retain={retain_score:.3f}")

    model.load_state_dict(best_state)
    for p in model.parameters():
        p.requires_grad = True
    print(f"\n[VGGFace2] Done. Best balance: {best_balance:.4f}")
    return model


In [ ]:
# ===========================================================================
# ▓  CELL I2: POST-UNLEARNING RETAIN REPAIR
# ===========================================================================

def run_retain_repair(
    model:              nn.Module,
    old_model:          nn.Module,
    retain_loader:      DataLoader,
    retain_test_loader: DataLoader,
    forget_ids:         List,
    num_classes:        int,
    cfg:                Config,
    retain_prototypes:  Optional[torch.Tensor] = None,   # [C, D] frozen, CPU
    retain_val_loader:  Optional[DataLoader]    = None,   # best-ckpt selection
    forget_loader:      Optional[DataLoader]    = None,   # guard chống un-erase
    forget_proto:       Optional[torch.Tensor]  = None,   # [D] frozen, CPU
    retain_acc_ref:     Optional[float]         = None,   # cho R3 catch-up
    beat_acc:           Optional[float]         = None,   # retain_accuracy cao nhất
                                                            # trong số baseline ĐÃ CHẠY (FineTune/
                                                            # NegGrad) — R3 nhắm target là
                                                            # max(retain_acc_ref, beat_acc).
) -> nn.Module:
    """
    Two-phase post-unlearning retain repair (+ R3 catch-up).

    Phase R1 (60% epochs, high LR): CE dominant, neck + retain-arcface +
      layer4 (lr nhỏ hơn) cùng adapt với embedding space mới.
    Phase R2 (40% epochs, low LR): KD-stabilisation dominant, tránh
      over-drift khỏi representation gốc.

    Best-checkpoint selection: theo dõi best state trên retain_val_loader
    (KHÔNG dùng retain_test_loader để chọn — tránh leak test set vào lựa
    chọn checkpoint), rồi load lại best_state trước khi trả về model.
    """
    device = cfg.device
    best_val_acc, best_state = -1.0, None
    model.to(device)
    old_model.eval()
    for p in old_model.parameters():
        p.requires_grad = False

    # VGGFace2: identity_equals_class=True → có 1 hàng ArcFace riêng cho
    # mỗi identity bị forget — đóng băng đúng các hàng đó (ở giá trị SAU
    # unlearning, KHÔNG phải giá trị gốc) trong suốt repair, để CE không
    # "học lại" identity đã bị quên.
    forget_cls_t = None
    stashed_forget_arcface = None
    if identity_equals_class(forget_ids, model.arcface.weight.shape[0]):
        fset = set(forget_ids)
        forget_cls_t = torch.tensor(list(fset), dtype=torch.long)
        stashed_forget_arcface = model.arcface.weight.data[forget_cls_t].clone()

    proto_dev = (
        retain_prototypes.to(device)
        if (cfg.use_proto_retain and retain_prototypes is not None) else None
    )

    # Guard chống un-erase — forget_iter độc lập với retain_loader, lặp
    # vòng qua forget_loader song song trong cả R1 và R2.
    guard_w = cfg.repair_forget_guard_weight
    forget_proto_dev = None
    forget_iter = None
    if forget_loader is not None and forget_proto is not None and guard_w > 0:
        forget_proto_dev = forget_proto.to(device)
        forget_iter = iter(forget_loader)

    def _guard_loss(model):
        """Lực nhỏ tiếp tục đẩy embedding forget ra xa old_forget_proto,
        song song với CE/KD trên retain — ngăn repair kéo neck/layer4
        trôi ngược về gần cấu hình gốc."""
        nonlocal forget_iter
        if forget_iter is None:
            return torch.zeros(1, device=device).squeeze()
        try:
            x_f, _, _ = next(forget_iter)
        except StopIteration:
            forget_iter = iter(forget_loader)
            x_f, _, _ = next(forget_iter)
        x_f = x_f.to(device)
        z_f, _ = model(x_f, None)
        return guard_w * (z_f * forget_proto_dev.unsqueeze(0)).sum(dim=1).mean()

    T  = cfg.kd_temperature
    ce = nn.CrossEntropyLoss()

    r1_epochs = max(1, int(cfg.repair_epochs * 0.6))
    r2_epochs = cfg.repair_epochs - r1_epochs

    print(f"\n[Repair] Two-phase retain repair — {cfg.repair_epochs} epochs total")
    print(f"         R1={r1_epochs}e (LR={cfg.repair_lr:.1e}, CE+KD, neck+retain-arcface adapt, "
          f"layer4 LR={cfg.repair_lr*cfg.repair_layer4_lr_mult:.1e})")
    print(f"         R2={r2_epochs}e (LR={cfg.repair_lr*0.1:.1e}, KD stabilisation)")
    print(f"         KEY: only FORGET arcface frozen; retain arcface + layer4 free to adapt")

    # ── PHASE R1: Adaptive neck + retain-arcface + layer4 re-alignment ──────
    trainable_r1 = [p for n, p in model.named_parameters()
                    if ("neck" in n or "arcface" in n) and "backbone.layer4" not in n]
    layer4_r1    = [p for n, p in model.named_parameters() if "backbone.layer4" in n]
    all_r1_params = trainable_r1 + layer4_r1
    opt_r1 = torch.optim.AdamW(
        [
            {"params": trainable_r1, "lr": cfg.repair_lr},
            {"params": layer4_r1,    "lr": cfg.repair_lr * cfg.repair_layer4_lr_mult},
        ],
        weight_decay=1e-5
    )
    sched_r1 = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt_r1, T_max=max(1, r1_epochs), eta_min=cfg.repair_lr * 0.1
    )

    for epoch in range(r1_epochs):
        model.train()
        total_loss = 0.0
        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)

            z_r, logits_new = model(x_r, y_r)

            loss_ce = cfg.w_ce_retain * ce(logits_new, y_r)

            with torch.no_grad():
                _, logits_old = old_model(x_r, y_r)
            log_p = F.log_softmax(logits_new / T, dim=1)
            p_old = F.softmax(logits_old   / T, dim=1)
            loss_kl = 5.0 * (T ** 2) * F.kl_div(log_p, p_old, reduction='batchmean')

            if proto_dev is not None:
                proto_y = proto_dev[y_r]
                loss_proto = cfg.w_proto_retain * (1.0 - (z_r * proto_y).sum(dim=1)).mean()
            else:
                loss_proto = torch.zeros(1, device=device).squeeze()

            loss_guard = _guard_loss(model)

            loss = loss_ce + loss_kl + loss_proto + loss_guard

            opt_r1.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(all_r1_params, 1.0)
            opt_r1.step()

            if forget_cls_t is not None:
                model.arcface.weight.data[forget_cls_t] = stashed_forget_arcface.to(device)

            total_loss += float(loss)

        sched_r1.step()
        val_source = retain_val_loader if retain_val_loader is not None else retain_test_loader
        ret_acc = evaluate_accuracy(model, val_source, device)
        if ret_acc > best_val_acc:
            best_val_acc = ret_acc
            best_state = copy.deepcopy(model.state_dict())
        print(f"[R1][{epoch+1:02d}/{r1_epochs}]  "
              f"loss={total_loss/len(retain_loader):.4f}  "
              f"retain_val_acc={ret_acc:.4f}  best={best_val_acc:.4f}")

    # ── PHASE R2: KD Stabilisation ────────────────────────────────────────────
    trainable_r2 = [p for n, p in model.named_parameters()
                    if ("neck" in n or "arcface" in n) and "backbone.layer4" not in n]
    layer4_r2    = [p for n, p in model.named_parameters() if "backbone.layer4" in n]
    all_r2_params = trainable_r2 + layer4_r2
    opt_r2 = torch.optim.AdamW(
        [
            {"params": trainable_r2, "lr": cfg.repair_lr * 0.1},
            {"params": layer4_r2,    "lr": cfg.repair_lr * 0.1 * cfg.repair_layer4_lr_mult},
        ],
        weight_decay=1e-5
    )

    for epoch in range(r2_epochs):
        model.train()
        total_loss = 0.0
        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)

            z_r, logits_new = model(x_r, y_r)
            with torch.no_grad():
                _, logits_old = old_model(x_r, y_r)

            log_p = F.log_softmax(logits_new / T, dim=1)
            p_old = F.softmax(logits_old   / T, dim=1)
            loss_kl = (cfg.w_kd_global * cfg.repair_r2_kd_mult
                       * (T ** 2) * F.kl_div(log_p, p_old, reduction='batchmean'))

            loss_ce = cfg.w_ce_retain * ce(logits_new, y_r)

            if proto_dev is not None:
                proto_y = proto_dev[y_r]
                loss_proto = (cfg.w_proto_retain * 0.5) * (1.0 - (z_r * proto_y).sum(dim=1)).mean()
            else:
                loss_proto = torch.zeros(1, device=device).squeeze()

            loss_guard = _guard_loss(model)

            loss = loss_kl + loss_ce + loss_proto + loss_guard

            opt_r2.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(all_r2_params, cfg.grad_clip_retain)
            opt_r2.step()

            if forget_cls_t is not None:
                model.arcface.weight.data[forget_cls_t] = stashed_forget_arcface.to(device)

            total_loss += float(loss)

        val_source = retain_val_loader if retain_val_loader is not None else retain_test_loader
        ret_acc = evaluate_accuracy(model, val_source, device)
        if ret_acc > best_val_acc:
            best_val_acc = ret_acc
            best_state = copy.deepcopy(model.state_dict())
        print(f"[R2][{epoch+1:02d}/{r2_epochs}]  "
              f"loss={total_loss/len(retain_loader):.4f}  "
              f"retain_val_acc={ret_acc:.4f}  best={best_val_acc:.4f}")

    # ── PHASE R3: CATCH-UP ──────────────────────────────────────────────
    if cfg.repair_catchup_enable and retain_acc_ref is not None:
        base_target = max(retain_acc_ref, beat_acc) if beat_acc is not None else retain_acc_ref
        target_acc = base_target * cfg.repair_catchup_target
        if best_val_acc < target_acc:
            print(f"\n[Repair] R3 catch-up: best_val_acc={best_val_acc:.4f} "
                  f"< target={target_acc:.4f} (retain_acc_ref×{cfg.repair_catchup_target}) "
                  f"— chạy thêm tối đa {cfg.repair_catchup_max_epochs} epoch")

            trainable_r3 = [p for n, p in model.named_parameters()
                             if ("neck" in n or "arcface" in n) and "backbone.layer4" not in n]
            layer4_r3     = [p for n, p in model.named_parameters() if "backbone.layer4" in n]
            all_r3_params = trainable_r3 + layer4_r3
            opt_r3 = torch.optim.AdamW(
                [
                    {"params": trainable_r3, "lr": cfg.repair_lr * 0.5},
                    {"params": layer4_r3,    "lr": cfg.repair_lr * 0.5 * cfg.repair_layer4_lr_mult},
                ],
                weight_decay=1e-5
            )

            catchup_epoch = 0
            while best_val_acc < target_acc and catchup_epoch < cfg.repair_catchup_max_epochs:
                catchup_epoch += 1
                model.train()
                total_loss = 0.0
                for x_r, y_r, _ in retain_loader:
                    x_r, y_r = x_r.to(device), y_r.to(device)
                    z_r, logits_new = model(x_r, y_r)

                    loss_ce = cfg.w_ce_retain * ce(logits_new, y_r)

                    with torch.no_grad():
                        _, logits_old = old_model(x_r, y_r)
                    log_p = F.log_softmax(logits_new / T, dim=1)
                    p_old = F.softmax(logits_old   / T, dim=1)
                    loss_kl = 5.0 * (T ** 2) * F.kl_div(log_p, p_old, reduction='batchmean')

                    if proto_dev is not None:
                        proto_y = proto_dev[y_r]
                        loss_proto = cfg.w_proto_retain * (1.0 - (z_r * proto_y).sum(dim=1)).mean()
                    else:
                        loss_proto = torch.zeros(1, device=device).squeeze()

                    loss_guard = _guard_loss(model)

                    loss = loss_ce + loss_kl + loss_proto + loss_guard

                    opt_r3.zero_grad()
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(all_r3_params, 1.0)
                    opt_r3.step()

                    if forget_cls_t is not None:
                        model.arcface.weight.data[forget_cls_t] = stashed_forget_arcface.to(device)

                    total_loss += float(loss)

                val_source = retain_val_loader if retain_val_loader is not None else retain_test_loader
                ret_acc = evaluate_accuracy(model, val_source, device)
                if ret_acc > best_val_acc:
                    best_val_acc = ret_acc
                    best_state = copy.deepcopy(model.state_dict())
                print(f"[R3][{catchup_epoch:02d}/{cfg.repair_catchup_max_epochs}]  "
                      f"loss={total_loss/len(retain_loader):.4f}  "
                      f"retain_val_acc={ret_acc:.4f}  best={best_val_acc:.4f}  target={target_acc:.4f}")

            if best_val_acc >= target_acc:
                print(f"[Repair] R3: đã bắt kịp target ({best_val_acc:.4f} >= {target_acc:.4f})")
            else:
                print(f"[Repair] R3: chạm trần {cfg.repair_catchup_max_epochs} epoch, "
                      f"best_val_acc={best_val_acc:.4f} vẫn < target={target_acc:.4f}")

    if best_state is not None:
        model.load_state_dict(best_state)
    final_acc = evaluate_accuracy(model, retain_test_loader, device)
    print(f"\n[Repair] Done. Best retain_val_acc={best_val_acc:.4f} → "
          f"Final retain_TEST_acc={final_acc:.4f}")
    return model


In [ ]:
# ===========================================================================
# ▓  CELL J: EVALUATION METRICS
# ===========================================================================

@torch.no_grad()
def evaluate_accuracy(model, loader, device):
    model.eval(); correct = total = 0
    for x, y, _ in loader:
        x, y = x.to(device), y.to(device)
        _, logits = model(x, None)
        correct += int((logits.argmax(1) == y).sum())
        total   += y.numel()
    return correct / max(1, total)

@torch.no_grad()
def mean_sim_to_proto(model, loader, proto, device):
    model.eval(); sims = []
    for x, _, _ in loader:
        z, _ = model(x.to(device), None)
        sims.append((z @ proto.to(device)).mean().item())
    return float(np.mean(sims)) if sims else 0.0

@torch.no_grad()
def mean_entropy(model, loader, device):
    model.eval(); vals = []
    for x, _, _ in loader:
        _, logits = model(x.to(device), None)
        p = F.softmax(logits, dim=1)
        vals.append(-(p * torch.log(p + 1e-8)).sum(1).mean().item())
    return float(np.mean(vals)) if vals else 0.0

@torch.no_grad()
def cluster_compactness(model, loader, device):
    model.eval()
    embs = [model(x.to(device), None)[0].cpu() for x, _, _ in loader]
    z = torch.cat(embs)
    return float(((z - z.mean(0, keepdim=True))**2).sum(1).mean())

@torch.no_grad()
def intra_cluster_cos_sim(model, loader, device):
    model.eval()
    embs = [model(x.to(device), None)[0].cpu() for x, _, _ in loader]
    z = F.normalize(torch.cat(embs), dim=1)
    if z.size(0) < 2: return 1.0
    sim_mat = z @ z.T
    mask = ~torch.eye(z.size(0), dtype=torch.bool)
    return float(sim_mat[mask].mean())

@torch.no_grad()
def neighbor_shift(new_m, old_m, loader, device):
    new_m.eval(); old_m.eval(); shifts = []
    for x, _, _ in loader:
        x = x.to(device)
        z_new, _ = new_m(x, None); z_old, _ = old_m(x, None)
        shifts.append((z_new - z_old).norm(dim=1).mean().item())
    return float(np.mean(shifts)) if shifts else 0.0

@torch.no_grad()
def get_max_confidence(model, loader, device, n_max=500):
    """Giữ lại để tương thích ngược — KHÔNG dùng cho MIA (xem get_loss_signal)."""
    model.eval(); scores = []
    for x, _, _ in loader:
        _, logits = model(x.to(device), None)
        scores.append(F.softmax(logits, dim=1).max(1).values.cpu())
        if sum(len(s) for s in scores) >= n_max: break
    return torch.cat(scores)[:n_max]

def get_loss_signal(model, loader, device, n_max=500):
    """
    MIA feature = per-sample CE loss (Yeom et al. 2018 — loss-based
    membership inference). KHÔNG dùng max-softmax-confidence: với
    arcface_s=32.0, softmax(logits*s) gần như luôn ~0.999+ một khi model
    hội tụ tốt (dù đúng hay sai) → mất hết phương sai phân biệt → AUC rơi
    về đúng 0.5 (bão hòa, không phải model đạt privacy hoàn hảo).
    """
    model.eval(); scores = []
    ce = nn.CrossEntropyLoss(reduction='none')
    with torch.no_grad():
        for x, y, _ in loader:
            x, y = x.to(device), y.to(device)
            _, logits = model(x, None)
            loss = ce(logits, y)
            scores.append((-loss).cpu())   # loss thấp (nhớ tốt) → score cao → "member"
            if sum(len(s) for s in scores) >= n_max: break
    return torch.cat(scores)[:n_max]

def _fit_mia_attacker(model, member_loader, nonmember_loader, device, cfg, n_bootstrap=30):
    """Cân bằng lớp member/non-member (cắt lớp nhiều hơn xuống bằng lớp ít
    hơn) TRƯỚC khi fit, cộng bootstrap resampling (30 lần, resample có hoàn
    lại từ chính k mẫu/lớp đã thu thập, fit lại logistic regression mỗi
    lần) để có độ lệch chuẩn đi kèm — nếu chênh lệch giữa 2 phương pháp nhỏ
    hơn ~1-2 lần std cộng lại, nên coi là KHÔNG phân biệt được về mặt thống
    kê, không phải "phương pháp A tốt hơn B".

    Trả về (auc, accuracy, auc_std, accuracy_std, n_per_class)."""
    n   = min(cfg.mia_n_members, cfg.mia_n_nonmembers)
    mem = get_loss_signal(model, member_loader,    device, n)
    non = get_loss_signal(model, nonmember_loader, device, n)
    k = min(len(mem), len(non))
    if k < 1:
        return None, None, None, None, 0
    mem_np = mem[:k].numpy()
    non_np = non[:k].numpy()

    def _fit_once(mem_s, non_s):
        X = np.concatenate([mem_s, non_s]).reshape(-1, 1)
        y = np.array([1] * len(mem_s) + [0] * len(non_s))
        if len(np.unique(y)) < 2:
            return None, None
        try:
            clf = LogisticRegression(max_iter=500)
            clf.fit(X, y)
            auc = float(roc_auc_score(y, clf.predict_proba(X)[:, 1]))
            acc = float(clf.score(X, y))
            return auc, acc
        except Exception:
            return None, None

    auc0, acc0 = _fit_once(mem_np, non_np)   # point estimate — fit trên đủ k mẫu/lớp
    if auc0 is None:
        return None, None, None, None, k
    if k < 2 or n_bootstrap <= 0:
        return auc0, acc0, 0.0, 0.0, k

    rng = np.random.RandomState(cfg.seed)
    aucs, accs = [], []
    for _ in range(n_bootstrap):
        mem_s = mem_np[rng.randint(0, k, size=k)]   # resample có hoàn lại
        non_s = non_np[rng.randint(0, k, size=k)]
        a, c = _fit_once(mem_s, non_s)
        if a is not None:
            aucs.append(a); accs.append(c)
    auc_std = float(np.std(aucs)) if aucs else 0.0
    acc_std = float(np.std(accs)) if accs else 0.0
    return auc0, acc0, auc_std, acc_std, k


def compute_mia_auc(model, forget_loader, retain_test_loader, device, cfg):
    auc, _, _, _, _ = _fit_mia_attacker(model, forget_loader, retain_test_loader, device, cfg)
    return auc if auc is not None else 0.5


def compute_mia_auc_unseen(model, forget_loader, unseen_loader, device, cfg):
    """Forgetting Score đúng định nghĩa Choi & Na (2023) / GLI (Choi et al.
    2024): so loss của x_forget với loss của x_unseen (danh tính CHƯA TỪNG
    được model gốc nhìn thấy), KHÔNG phải x_retain_test. Trả về AUC — giữ
    lại để tham khảo, KHÔNG dùng để so trực tiếp với Forgetting Score
    trong 2 bài báo (paper dùng accuracy — xem compute_mia_accuracy_unseen)."""
    if unseen_loader is None:
        return None
    auc, _, _, _, _ = _fit_mia_attacker(model, forget_loader, unseen_loader, device, cfg)
    return auc if auc is not None else 0.5


def compute_mia_accuracy_unseen(model, forget_loader, unseen_loader, device, cfg):
    """ĐÚNG 100% công thức Forgetting Score trong paper gốc. Trích Choi &
    Na (2023): "M denotes the accuracy of ψ(·)". Trích GLI: "forgetting
    score as abs(0.5 − M) where M denotes the ACCURACY of the MIA model"
    (không phải AUC)."""
    if unseen_loader is None:
        return None
    _, acc, _, _, _ = _fit_mia_attacker(model, forget_loader, unseen_loader, device, cfg)
    return acc if acc is not None else 0.5


def full_evaluate(
    name: str,
    model: nn.Module,
    old_model: nn.Module,
    retain_test_loader: DataLoader,
    forget_eval_loader: DataLoader,
    local_loader: DataLoader,
    old_proto: torch.Tensor,
    cfg: Config,
    num_classes: int,
    unseen_loader: Optional[DataLoader] = None,
) -> Dict:
    device = cfg.device
    rc     = 1.0 / num_classes

    ret_acc  = evaluate_accuracy(model, retain_test_loader, device)
    fgt_acc  = evaluate_accuracy(model, forget_eval_loader, device)
    old_sim  = mean_sim_to_proto(old_model, forget_eval_loader, old_proto, device)
    new_sim  = mean_sim_to_proto(model,     forget_eval_loader, old_proto, device)
    nshift   = neighbor_shift(model, old_model, local_loader, device)
    old_ent  = mean_entropy(old_model, forget_eval_loader, device)
    new_ent  = mean_entropy(model,     forget_eval_loader, device)
    old_comp = cluster_compactness(old_model, forget_eval_loader, device)
    new_comp = cluster_compactness(model,     forget_eval_loader, device)
    old_ics  = intra_cluster_cos_sim(old_model, forget_eval_loader, device)
    new_ics  = intra_cluster_cos_sim(model,     forget_eval_loader, device)
    mia      = compute_mia_auc(model, forget_eval_loader, retain_test_loader, device, cfg)
    _mia_auc_u, _mia_acc_u, _mia_auc_std, _mia_acc_std, _mia_n_per_class = _fit_mia_attacker(
        model, forget_eval_loader, unseen_loader, device, cfg
    ) if unseen_loader is not None else (None, None, None, None, 0)
    mia_unseen     = _mia_auc_u
    mia_acc_unseen = _mia_acc_u

    m = dict(
        retain_accuracy       = ret_acc,
        forget_accuracy       = fgt_acc,
        forget_acc_vs_random  = fgt_acc - rc,
        old_forget_sim        = old_sim,
        new_forget_sim        = new_sim,
        forget_sim_drop       = old_sim - new_sim,
        neighbor_shift        = nshift,
        old_entropy           = old_ent,
        new_entropy           = new_ent,
        entropy_increase      = new_ent - old_ent,
        old_compactness       = old_comp,
        new_compactness       = new_comp,
        compactness_increase  = new_comp - old_comp,
        old_intra_cluster_sim = old_ics,
        new_intra_cluster_sim = new_ics,
        cluster_sim_drop      = old_ics - new_ics,
        mia_auc               = mia,
        mia_auc_unseen        = mia_unseen,
        forgetting_score        = abs(mia - 0.5),
        forgetting_score_unseen = abs(mia_unseen - 0.5) if mia_unseen is not None else None,
        mia_accuracy_unseen           = mia_acc_unseen,
        forgetting_score_unseen_paper = abs(mia_acc_unseen - 0.5) if mia_acc_unseen is not None else None,
        mia_n_per_class                = _mia_n_per_class,
        mia_accuracy_unseen_std        = _mia_acc_std,
        forgetting_score_unseen_paper_std = _mia_acc_std,
    )

    bar = "═" * 70
    tgt_fgt = "✓" if abs(fgt_acc - rc) < 0.01 else ("⚠ low"  if fgt_acc < rc*0.5 else "⚠ high")
    tgt_sim = "✓" if (old_sim - new_sim) > 0.25  else "·"
    tgt_ns  = "✓" if nshift < 0.15               else "⚠"
    tgt_ent = "✓" if (new_ent - old_ent) > 0.5   else "·"
    tgt_mia = "✓" if abs(mia - 0.5) < 0.03       else "·"
    tgt_ics = "✓" if (old_ics - new_ics) > 0.20  else "·"

    print(f"\n{bar}")
    print(f"  [{name}]")
    print(f"{bar}")
    print(f"  Retain Accuracy      : {ret_acc:.4f}          ← HIGH is good")
    print(f"  Forget Accuracy      : {fgt_acc:.4f}  {tgt_fgt}  ← ~{rc:.4f} ideal (1/C)")
    print(f"  Forget Sim Drop  ↑   : {old_sim-new_sim:+.4f}  {tgt_sim}  ← ≥ +0.25 target")
    print(f"  Cluster Cos Drop ↑   : {old_ics-new_ics:+.4f}  {tgt_ics}  ← ≥ +0.20 target")
    print(f"  Compactness Incr ↑   : {new_comp-old_comp:+.4f}          ← positive = dispersed")
    print(f"  Entropy Increase ↑   : {new_ent-old_ent:+.4f}  {tgt_ent}  ← ≥ +0.50 target")
    print(f"  Neighbor Shift   ↓   : {nshift:.4f}  {tgt_ns}  ← < 0.15 target")
    print(f"  MIA AUC (vs retain)  : {mia:.4f}  {tgt_mia}  ← 0.5 ideal (tín hiệu forget-vs-retain, KHÔNG so được với Choi/GLI)")
    if mia_unseen is not None:
        tgt_mia_u = "✓" if abs(mia_unseen - 0.5) < 0.03 else "·"
        print(f"  MIA AUC (vs unseen)  : {mia_unseen:.4f}  {tgt_mia_u}  ← 0.5 ideal — ĐÚNG chuẩn Forgetting Score Choi & Na / GLI")
        print(f"  ┌─ Forgetting Score (AUC-based, chỉ tham khảo)  : {abs(mia_unseen-0.5):.4f}")
        tgt_mia_acc = "✓" if abs(mia_acc_unseen - 0.5) < 0.03 else "·"
        print(f"  MIA Accuracy (vs unseen)  : {mia_acc_unseen:.4f}  {tgt_mia_acc}  ← 0.5 ideal — ĐÚNG M trong paper (accuracy, không phải AUC)")
        print(f"  └─ FORGETTING SCORE (đúng công thức paper: |0.5-accuracy|) : {abs(mia_acc_unseen-0.5):.4f}  ← DÙNG SỐ NÀY để so với GLI/Choi&Na")
        print(f"     (n={_mia_n_per_class} mẫu/lớp, đã cân bằng. Nếu n rất nhỏ (<15), "
              f"kết quả accuracy dễ nhiễu — cân nhắc tăng num_forget_ids hoặc đọc thêm forgetting_score_unseen làm tham khảo chéo)")
        print(f"     ± {_mia_acc_std:.4f} (std, bootstrap {30 if _mia_n_per_class >= 2 else 0} lần). "
              f"So 2 phương pháp: nếu |chênh lệch FS| < tổng 2 std, coi là KHÔNG khác biệt có ý nghĩa thống kê.")
    else:
        print(f"  MIA AUC (vs unseen)  : (chưa truyền unseen_loader)")
    print(f"{bar}")
    return m


In [ ]:
# ===========================================================================
# ▓  CELL J2: RE-IDENTIFICATION LEAKAGE METRIC (embedding retrieval attack)
# ===========================================================================
#
# forget_accuracy (Cell J) đo ở CẤP TASK (phân loại identity) — với
# VGGFace2, identity CHÍNH LÀ class (identity_equals_class=True), nên
# forget_accuracy CÓ THỂ giảm thật (khác MUFAC, nơi đầu ArcFace luôn bị
# đóng băng/khôi phục nên forget_accuracy không đổi được — xem companion
# paper, Section "Discussion"). Metric dưới đây đo "quên" ở CẤP EMBEDDING —
# mô phỏng trực tiếp threat model mở đầu bài báo LPEU: một kẻ tấn công có
# quyền truy cập embedding (không cần đầu phân loại) tìm kiếm hàng-xóm-
# gần-nhất (nearest-neighbour retrieval) để NHẬN DIỆN LẠI người đã bị
# "quên", giống hệ thống face-retrieval thật.

@torch.no_grad()
def compute_reidentification_accuracy(
    model: nn.Module,
    eval_base,
    forget_person_to_idx: Dict,
    retain_person_to_idx: Dict,
    cfg: Config,
) -> Dict:
    device = cfg.device
    model.eval()

    def _embed_indices(idx_list):
        if not idx_list:
            return torch.empty(0, cfg.embedding_dim)
        ds = IndexedSubset(eval_base, idx_list)
        loader = make_loader(ds, cfg.batch_size, False, cfg.num_workers)
        embs = []
        for x, _, _ in loader:
            z, _ = model(x.to(device), None)
            embs.append(z.cpu())
        return torch.cat(embs)

    labels, protos, forget_embs = [], [], {}
    for pkey, idxs in retain_person_to_idx.items():
        if not idxs:
            continue
        e = _embed_indices(idxs)
        protos.append(F.normalize(e.mean(0, keepdim=True), dim=1))
        labels.append(pkey)
    for pkey, idxs in forget_person_to_idx.items():
        if not idxs:
            continue
        e = _embed_indices(idxs)
        forget_embs[pkey] = e
        protos.append(F.normalize(e.mean(0, keepdim=True), dim=1))
        labels.append(pkey)

    gallery = torch.cat(protos, dim=0)                          # [N, D]
    N = gallery.size(0)
    label_to_row = {lbl: i for i, lbl in enumerate(labels)}      # 1 identity = 1 hàng

    n_skipped = 0
    ranks = []                        # hạng (1-indexed) của đúng identity, theo similarity giảm dần
    top1 = top5 = top10 = 0
    total = 0
    for pkey, e_all in forget_embs.items():
        k = e_all.size(0)
        if k < 2:
            n_skipped += 1
            continue
        self_row = label_to_row[pkey]
        for i in range(k):
            mask = torch.ones(k, dtype=torch.bool); mask[i] = False
            loo_proto = F.normalize(e_all[mask].mean(0, keepdim=True), dim=1)   # [1, D]
            g = gallery.clone()
            g[self_row] = loo_proto.squeeze(0)          # chỉ thay đúng 1 hàng — không rò rỉ x_i
            query = F.normalize(e_all[i:i+1], dim=1)
            sims = (query @ g.T).squeeze(0)              # [N]
            order = torch.argsort(sims, descending=True)
            rank = int((order == self_row).nonzero(as_tuple=True)[0].item()) + 1
            ranks.append(rank)
            total += 1
            if rank == 1:  top1  += 1
            if rank <= 5:  top5  += 1
            if rank <= 10: top10 += 1

    ranks_t = torch.tensor(ranks, dtype=torch.float32) if ranks else torch.empty(0)
    return dict(
        reid_accuracy                       = top1 / max(1, total),
        reid_top5_accuracy                  = top5 / max(1, total),
        reid_top10_accuracy                 = top10 / max(1, total),
        reid_mean_rank                      = float(ranks_t.mean()) if len(ranks) else None,
        reid_median_rank                    = float(ranks_t.median()) if len(ranks) else None,
        reid_chance_mean_rank               = (N + 1) / 2.0,
        reid_n_queries                      = total,
        reid_gallery_size                   = N,
        reid_n_persons_evaluated            = len(forget_embs) - n_skipped,
        reid_n_persons_skipped_single_image = n_skipped,
    )


In [ ]:
# ===========================================================================
# ▓  CELL K: BASELINES — nguồn tham khảo (không tự phát minh)
# ===========================================================================
#
# [1] Fine-tuning-on-retain: Golatkar, Achille, Soatto. "Eternal Sunshine of
#     the Spotless Net: Selective Forgetting in Deep Networks." CVPR 2020.
#     arXiv:1911.04933.
# [2] NegGrad / NegGrad+ (gradient ascent trên forget set): cùng nguồn trên;
#     công thức joint loss (loss_r − loss_f) formal hoá trong Kurmanji et al.
#     (SCRUB), NeurIPS 2023. arXiv:2302.09880.
# [3] Choi, Choi, Lee, Seo, Na. "Towards Efficient Machine Unlearning with
#     Data Augmentation: Guided Loss-Increasing (GLI)." CVPR Workshops 2024
#     — cùng nhóm tác giả với MUFAC (Dongbin Na, POSTECH), cùng dùng
#     Fine-tuning + NegGrad làm baseline.
# ===========================================================================

def run_finetune(model, retain_loader, cfg, num_epochs=5):
    """Baseline: Fine-tuning-on-retain."""
    device = cfg.device; model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=1e-4, momentum=0.9, weight_decay=1e-4)
    ce  = nn.CrossEntropyLoss()
    for _ in range(num_epochs):
        model.train()
        for x, y, _ in retain_loader:
            x, y = x.to(device), y.to(device)
            _, logits = model(x, y)
            loss = ce(logits, y)
            opt.zero_grad(); loss.backward(); opt.step()
    return model


def run_neggrad(model, forget_loader, retain_loader, cfg, num_epochs=5):
    """Baseline: NegGrad / NegGrad+ (gradient ascent trên forget set)."""
    device = cfg.device; model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=1e-5, momentum=0.9)
    ce  = nn.CrossEntropyLoss()
    forget_iter = iter(forget_loader)
    for _ in range(num_epochs):
        model.train()
        for x_r, y_r, _ in retain_loader:
            x_r, y_r = x_r.to(device), y_r.to(device)
            _, logits_r = model(x_r, y_r); loss_r = ce(logits_r, y_r)
            try: x_f, y_f, _ = next(forget_iter)
            except StopIteration:
                forget_iter = iter(forget_loader)
                x_f, y_f, _ = next(forget_iter)
            x_f, y_f = x_f.to(device), y_f.to(device)
            _, logits_f = model(x_f, y_f)
            loss = loss_r - ce(logits_f, y_f)
            opt.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
    return model


In [ ]:
import os
for f in ["original_model.pt", "lpeu_vggface2_model.pt", "lpeu_vggface2_repaired_model.pt", "results_vggface2_C.json"]:
    p = os.path.join("./runs/lpeu_v7_vggface2_C", f)
    if os.path.exists(p):
        os.remove(p)
        print("Deleted:", p)


In [ ]:
# ===========================================================================
# ▓  CELL L: FULL PIPELINE (VGGFace2 — Hướng C)
# ===========================================================================

# ─── Step 1: Dataset ─────────────────────────────────────────────────────────
print("\n[Step 1] Loading dataset (VGGFace2)...")
train_base, eval_base, filtered_indices, valid_classes = load_full_dataset(cfg)
num_classes = len(valid_classes)
split = split_by_identity(train_base, filtered_indices, cfg)

# v_reid — gom sẵn ảnh-train theo từng IDENTITY (forget & retain) để dùng
# cho metric nhận diện lại (re-identification) ở Step 7b bên dưới. Ở
# VGGFace2, "khoá cá nhân" CHÍNH LÀ nhãn lớp (int) — khác MUFAC (chuỗi
# "family_id::person_id") vì ở đây identity == class.
forget_person_to_idx: Dict[int, List[int]] = {}
for _idx in split["forget_idx"]:
    _y = train_base.samples[_idx][1]
    forget_person_to_idx.setdefault(_y, []).append(_idx)

retain_person_to_idx: Dict[int, List[int]] = {}
for _idx in split["retain_train_idx"]:
    _y = train_base.samples[_idx][1]
    retain_person_to_idx.setdefault(_y, []).append(_idx)

print(f"  Identities: {num_classes}  |  Forget IDs: {len(split['forget_ids'])}")
print(f"  Forget train: {len(split['forget_idx'])}  |  "
      f"Retain train: {len(split['retain_train_idx'])}  |  "
      f"Retain test: {len(split['retain_test_idx'])}")

# ─── Step 2: Loaders ─────────────────────────────────────────────────────────
train_ds          = IndexedSubset(train_base, split["train_idx"])
val_ds            = IndexedSubset(eval_base,  split["val_idx"])
retain_train_ds   = IndexedSubset(train_base, split["retain_train_idx"])
retain_test_ds    = IndexedSubset(eval_base,  split["retain_test_idx"])
retain_val_ds     = IndexedSubset(eval_base,  split["retain_val_idx"])
forget_train_ds   = IndexedSubset(train_base, split["forget_idx"])
# v7.9-fix7 — ĐÚNG GIAO THỨC PAPER: Dforget = phần ảnh nằm trong Dtrain mà
# model gốc đã học (Choi & Na 2023: "Dtrain = Dforget ∪ Dretain"), KHÔNG
# phải một phần test tách riêng — forget_idx (70% theo train_ratio) có
# nhiều ảnh hơn forget_test_idx (20%), giảm nhiễu MIA.
forget_eval_ds    = IndexedSubset(eval_base,  split["forget_idx"])

train_loader        = make_loader(train_ds,       cfg.batch_size, True,  cfg.num_workers)
val_loader          = make_loader(val_ds,          cfg.batch_size, False, cfg.num_workers)
retain_train_loader = make_loader(retain_train_ds, cfg.batch_size, True,  cfg.num_workers)
retain_test_loader  = make_loader(retain_test_ds,  cfg.batch_size, False, cfg.num_workers)
retain_val_loader   = make_loader(retain_val_ds,   cfg.batch_size, False, cfg.num_workers)
forget_train_loader = make_loader(forget_train_ds, cfg.batch_size, True,  cfg.num_workers)
forget_eval_loader  = make_loader(forget_eval_ds,  cfg.batch_size, False, cfg.num_workers)

# D_unseen: identity chưa từng được model gốc nhìn thấy, dùng để tính
# Forgetting Score đúng chuẩn Choi & Na / GLI.
unseen_ds     = IndexedSubset(eval_base, split["unseen_idx"])
unseen_loader = make_loader(unseen_ds, cfg.batch_size, False, cfg.num_workers)
print(f"  Unseen identities: {len(split['unseen_classes'])}  |  Unseen images: {len(split['unseen_idx'])}")

# Hướng C: pool ảnh unseen dùng cho L_div (xem _sample_unseen_batch_vgg,
# Cell I). KHÔNG class-matched (khác MUFAC — xem ghi chú ở Cell B/I): lấy
# NGẪU NHIÊN tối đa cfg.div_unseen_pool_max ảnh từ D_unseen, stack thành 1
# tensor duy nhất — giữ trong RAM suốt vòng lặp unlearning. eval_base
# (KHÔNG augment) để p_orig_unseen ổn định.
_unseen_pool_idx = list(split["unseen_idx"])
random.Random(cfg.seed).shuffle(_unseen_pool_idx)
_unseen_pool_idx = _unseen_pool_idx[:cfg.div_unseen_pool_max]
unseen_pool_images = torch.stack([eval_base[_i][0] for _i in _unseen_pool_idx])  # [N,C,H,W], CPU
print(f"  [Hướng C] Unseen pool cho L_div (không class-matched): "
      f"{unseen_pool_images.size(0)} ảnh (trần={cfg.div_unseen_pool_max})")

# ─── Step 3: Train or load original model ────────────────────────────────────
print("\n[Step 2] Training original model  [arcface_s=32, m=0.30, Simple Neck]...")
original_model = FaceModel(
    num_classes=num_classes,
    embedding_dim=cfg.embedding_dim,
    pretrained=cfg.pretrained_backbone,
    s=cfg.arcface_s, m=cfg.arcface_m,
)

orig_ckpt = os.path.join(cfg.output_dir, "original_model.pt")
if os.path.exists(orig_ckpt):
    print(f"  → Loading cached model from {orig_ckpt}")
    original_model.load_state_dict(torch.load(orig_ckpt, map_location=cfg.device))
    original_model.to(cfg.device)
else:
    original_model = train_original_model(original_model, train_loader, val_loader, cfg)
    torch.save(original_model.state_dict(), orig_ckpt)
    print(f"  → Saved to {orig_ckpt}")

# ─── Step 4: Prototypes & K-NN anchor ────────────────────────────────────────
print("\n[Step 3] Computing prototypes & K-NN anchor...")
all_prototypes = compute_class_prototypes(
    original_model,
    make_loader(IndexedSubset(eval_base, split["train_idx"]),
                cfg.batch_size, False, cfg.num_workers),
    num_classes, cfg.device,
)
old_forget_proto = compute_forget_prototype(original_model, forget_eval_loader, cfg.device)

# K-NN "local identity": ở VGGFace2, group == classifier class, nên
# get_identity_group_ids/compute_group_prototypes (Cell F, generic) cho ra
# đúng CÙNG kết quả như all_prototypes ở trên — dùng bản group-based để
# nhất quán mã nguồn với MUFAC-C (và để gradient projection, vốn cần một
# ma trận [K,D] tách biệt khỏi retain_prototypes đầy đủ, hoạt động đúng).
identity_group_ids = get_identity_group_ids(eval_base)
group_prototypes = compute_group_prototypes(
    original_model,
    make_loader(IndexedSubset(eval_base, split["train_idx"]),
                cfg.batch_size, False, cfg.num_workers),
    identity_group_ids, cfg.device,
)
anchor_group_ids = find_knn_retain_groups(
    forget_proto=old_forget_proto,
    group_prototypes=group_prototypes,
    forget_group_ids=split["forget_ids"],   # VGGFace2: forget_ids CHÍNH LÀ group id (class)
    k=cfg.k_anchor,
)
print(f"  Forget prototype ‖p‖ = {old_forget_proto.norm():.4f}")
print(f"  K-NN anchor identities: {anchor_group_ids}")

# Ma trận [K, D] local retain-prototype, tính từ original_model (frozen)
# TRƯỚC unlearning — dùng cho gradient projection.
local_retain_prototypes = (
    torch.stack([group_prototypes[g] for g in anchor_group_ids])
    if anchor_group_ids else None
)

# Verify intra_cluster_sim — key health check for the model
ics = intra_cluster_cos_sim(original_model, forget_eval_loader, cfg.device)
fsp = mean_sim_to_proto(original_model, forget_eval_loader, old_forget_proto, cfg.device)
print(f"\n  ── Model Health Check ──────────────────────────────────")
print(f"  old_forget_sim       = {fsp:.4f}   (target: 0.5–0.8)")
print(f"  intra_cluster_sim    = {ics:.4f}   (target: 0.30–0.60)")
if ics > 0.80:
    print(f"  ⚠ intra_cluster_sim TOO HIGH ({ics:.3f} > 0.80)")
    print(f"    This means embedding collapse. The model will be hard to unlearn.")
    print(f"    → Delete the cached model and retrain.")
else:
    print(f"  ✓ intra_cluster_sim looks healthy")
print(f"  ──────────────────────────────────────────────────────")

anchor_loader = build_local_anchor_loader_grouped(
    eval_base, split["retain_train_idx"], identity_group_ids,
    anchor_group_ids, cfg.batch_size, cfg.num_workers,
)

_anchor_group_set = set(anchor_group_ids)
local_neighbor_loader = make_loader(
    IndexedSubset(eval_base, [
        i for i in split["retain_train_idx"]
        if identity_group_ids[i] in _anchor_group_set
    ]),
    cfg.batch_size, False, cfg.num_workers,
)

# ─── Step 5: EWC Fisher computation ──────────────────────────────────────────
print("\n[Step 4] Computing EWC Fisher Information...")
fisher = theta_star = None
if cfg.use_ewc:
    theta_star = snapshot_neck_params(original_model)
    fisher     = compute_ewc_fisher(
        original_model, retain_train_loader, cfg.device, n_batches=30
    )
else:
    print("  EWC disabled (use_ewc=False).")

# ─── Step 6: Baselines ───────────────────────────────────────────────────────
print("\n[Step 5] Running baselines...")
ft_model = run_finetune(clone_model(original_model), retain_train_loader, cfg)
ng_model = run_neggrad(clone_model(original_model), forget_train_loader, retain_train_loader, cfg)

# Tính sớm retain_accuracy của baseline để LPEU repair (Step 6b) nhắm
# target là con số CAO NHẤT thực sự sẽ xuất hiện trong bảng kết quả.
ft_retain_acc = evaluate_accuracy(ft_model, retain_test_loader, cfg.device)
ng_retain_acc = evaluate_accuracy(ng_model, retain_test_loader, cfg.device)
best_baseline_retain_acc = max(ft_retain_acc, ng_retain_acc)
print(f"  FineTune retain_accuracy: {ft_retain_acc:.4f}  |  "
      f"NegGrad retain_accuracy: {ng_retain_acc:.4f}  |  "
      f"beat target for LPEU: {best_baseline_retain_acc:.4f}")

# ─── Step 7: LPEU ─────────────────────────────────────────────────────────
print("\n[Step 6] Running LPEU (VGGFace2, Hướng C)...")
retain_acc_ref = evaluate_accuracy(original_model, retain_test_loader, cfg.device)
print(f"  Reference retain accuracy: {retain_acc_ref:.4f}")

lpeu_model = run_lpeu_v7_unlearning(
    model               = clone_model(original_model),
    old_model           = clone_model(original_model),
    forget_loader       = forget_train_loader,
    retain_loader       = retain_train_loader,
    retain_test_loader  = retain_test_loader,
    anchor_loader       = anchor_loader,
    forget_ids          = split["forget_ids"],
    num_classes         = num_classes,
    cfg                 = cfg,
    retain_acc_ref      = retain_acc_ref,
    fisher              = fisher,
    theta_star          = theta_star,
    retain_prototypes   = all_prototypes,
    local_retain_prototypes = local_retain_prototypes,
    unseen_pool_images      = unseen_pool_images,   # Hướng C: L_div
)
torch.save(lpeu_model.state_dict(),
           os.path.join(cfg.output_dir, "lpeu_vggface2_model.pt"))
print("Saved LPEU model (before repair).")

# DIAGNOSTIC: eval NGAY TRƯỚC khi repair chạy, để tách bạch forgetting đạt
# được bởi Migrate→Shatter khỏi ảnh hưởng NGƯỢC của repair.
_eval_kwargs_prerepair = dict(
    old_model          = original_model,
    retain_test_loader = retain_test_loader,
    forget_eval_loader = forget_eval_loader,
    local_loader       = local_neighbor_loader,
    old_proto          = old_forget_proto,
    cfg                = cfg,
    num_classes        = num_classes,
    unseen_loader      = unseen_loader,
)
lp_prerepair_m = full_evaluate("LPEU-VGGFace2 (pre-repair)", lpeu_model, **_eval_kwargs_prerepair)
print(f"  [pre-repair snapshot] retain_acc={lp_prerepair_m['retain_accuracy']:.4f}  "
      f"forget_acc={lp_prerepair_m['forget_accuracy']:.4f}  "
      f"cluster_sim_drop={lp_prerepair_m['cluster_sim_drop']:.4f}  "
      f"mia_auc_unseen={lp_prerepair_m['mia_auc_unseen']:.4f}")

print("\n[Step 7c] Re-identification — LPEU checkpoint TRƯỚC repair (đối chứng)...")
reid_lp_prerepair = compute_reidentification_accuracy(
    lpeu_model, eval_base, forget_person_to_idx, retain_person_to_idx, cfg
)
lp_prerepair_m.update(reid_lp_prerepair)
print(f"  [pre-repair reid] top1={reid_lp_prerepair['reid_accuracy']:.4f}  "
      f"top5={reid_lp_prerepair['reid_top5_accuracy']:.4f}  "
      f"top10={reid_lp_prerepair['reid_top10_accuracy']:.4f}  "
      f"mean_rank={reid_lp_prerepair['reid_mean_rank']:.1f}  "
      f"median_rank={reid_lp_prerepair['reid_median_rank']:.1f}  "
      f"(chance≈{reid_lp_prerepair['reid_chance_mean_rank']:.1f})")

# ─── Post-unlearning retain repair ───────────────────────────────────────────
# CHỈ chạy repair nếu retain TRƯỚC repair thực sự thấp hơn tham chiếu (có
# "hỏng" thật cần vá) — nếu không, bỏ qua hẳn bước repair, giữ nguyên
# model ngay sau Migrate→Shatter (retain đã đủ, forgetting mạnh hơn).
_needs_repair = cfg.use_post_repair and (lp_prerepair_m['retain_accuracy'] < retain_acc_ref)
if _needs_repair:
    print("\n[Step 6b] Post-unlearning retain repair...")
    lpeu_model = run_retain_repair(
        model               = lpeu_model,
        old_model           = clone_model(original_model),
        retain_loader       = retain_train_loader,
        retain_test_loader  = retain_test_loader,
        forget_ids          = split["forget_ids"],
        num_classes         = num_classes,
        cfg                 = cfg,
        retain_prototypes   = all_prototypes,
        retain_val_loader   = retain_val_loader,
        forget_loader       = forget_train_loader,
        forget_proto        = old_forget_proto,
        retain_acc_ref      = retain_acc_ref,
        beat_acc            = best_baseline_retain_acc,
    )
    torch.save(lpeu_model.state_dict(),
               os.path.join(cfg.output_dir, "lpeu_vggface2_repaired_model.pt"))
    print("Saved LPEU repaired model.")
else:
    print(f"\n[Step 6b] Bỏ qua repair — retain trước repair "
          f"({lp_prerepair_m['retain_accuracy']:.4f}) đã ≥ retain_acc_ref "
          f"({retain_acc_ref:.4f}); repair chỉ tốn thêm forgetting cho retain "
          f"gần như không đổi. Model LPEU giữ nguyên trạng thái ngay sau "
          f"Migrate→Shatter.")

# ─── Step 8: Evaluation ──────────────────────────────────────────────────────
print("\n[Step 7] Full evaluation...")

eval_kwargs = dict(
    old_model          = original_model,
    retain_test_loader = retain_test_loader,
    forget_eval_loader = forget_eval_loader,
    local_loader       = local_neighbor_loader,
    old_proto          = old_forget_proto,
    cfg                = cfg,
    num_classes        = num_classes,
    unseen_loader      = unseen_loader,
)

orig_m = full_evaluate("Original",       original_model, **eval_kwargs)
ft_m   = full_evaluate("FineTune",       ft_model,       **eval_kwargs)
ng_m   = full_evaluate("NegGrad",        ng_model,       **eval_kwargs)
lp_m   = full_evaluate("LPEU-VGGFace2 ★", lpeu_model,    **eval_kwargs)

# ─── Step 8b: Re-identification leakage (embedding retrieval attack) ────────
print("\n[Step 7b] Re-identification (embedding retrieval)...")
reid_orig = compute_reidentification_accuracy(original_model, eval_base, forget_person_to_idx, retain_person_to_idx, cfg)
reid_ft   = compute_reidentification_accuracy(ft_model,        eval_base, forget_person_to_idx, retain_person_to_idx, cfg)
reid_ng   = compute_reidentification_accuracy(ng_model,        eval_base, forget_person_to_idx, retain_person_to_idx, cfg)
reid_lp   = compute_reidentification_accuracy(lpeu_model,      eval_base, forget_person_to_idx, retain_person_to_idx, cfg)

for _m, _r in [(orig_m, reid_orig), (ft_m, reid_ft), (ng_m, reid_ng), (lp_m, reid_lp)]:
    _m.update(_r)
    _m["reid_accuracy_drop_vs_original"] = reid_orig["reid_accuracy"] - _r["reid_accuracy"]

_bar2 = "═" * 70
print(f"\n{_bar2}")
print("  RE-IDENTIFICATION — attacker còn nhận ra người bị quên qua embedding?")
print(f"{_bar2}")
for _name, _r in [("Original", reid_orig), ("FineTune", reid_ft), ("NegGrad", reid_ng), ("LPEU", reid_lp)]:
    print(f"  {_name:10s}: top1={_r['reid_accuracy']:.4f}  top5={_r['reid_top5_accuracy']:.4f}  "
          f"top10={_r['reid_top10_accuracy']:.4f}  mean_rank={_r['reid_mean_rank']:.1f}  "
          f"median_rank={_r['reid_median_rank']:.1f}  (chance≈{_r['reid_chance_mean_rank']:.1f})")
    print(f"             (n={_r['reid_n_queries']} truy vấn, {_r['reid_n_persons_evaluated']} người, "
          f"gallery={_r['reid_gallery_size']} người, bỏ qua {_r['reid_n_persons_skipped_single_image']} người chỉ có 1 ảnh)")
print(f"  ↓ top-k càng thấp / rank càng CAO = quên càng tốt (khó nhận diện lại).")
print(f"{_bar2}")

results = dict(
    original       = orig_m,
    finetune       = ft_m,
    neggrad        = ng_m,
    lpeu           = lp_m,
    lpeu_prerepair = lp_prerepair_m,
    forget_ids     = split["forget_ids"],
    config = dict(
        embedding_dim  = cfg.embedding_dim,
        arcface_s      = cfg.arcface_s,
        arcface_m      = cfg.arcface_m,
        k_anchor       = cfg.k_anchor,
        w_kd_global    = cfg.w_kd_global,
        w_kd_local     = cfg.w_kd_local,
        w_ewc          = cfg.w_ewc,
        w_erase        = cfg.w_erase,
        w_kl_uniform   = cfg.w_kl_uniform,
        w_div          = cfg.w_div,   # Hướng C — để chẩn đoán: xem L_div có thật sự được cấu hình bật không
        w_repel        = cfg.w_repel,
        repel_margin   = cfg.repel_margin,
        pcgrad_boost   = cfg.pcgrad_boost,
        stage1_epochs  = cfg.stage1_epochs,
        stage2_epochs  = cfg.stage2_epochs,
        stage3_epochs  = cfg.stage3_epochs,
        w_proto_retain = cfg.w_proto_retain,
        original_epochs      = cfg.original_epochs,
        early_stop_patience  = cfg.early_stop_patience,
        safety_grace_epochs  = cfg.safety_grace_epochs,
        safety_patience      = cfg.safety_patience,
        forget_manual_lr     = cfg.forget_manual_lr,
        repair_layer4_lr_mult = cfg.repair_layer4_lr_mult,
        repair_r2_kd_mult     = cfg.repair_r2_kd_mult,
        num_forget_ids        = cfg.num_forget_ids,
        div_unseen_pool_max   = cfg.div_unseen_pool_max,
    )
)
out_path = os.path.join(cfg.output_dir, "results_vggface2_C.json")
with open(out_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"\nSaved results → {out_path}")


In [ ]:
# ===========================================================================
# ▓  CELL M: ABLATION STUDY
# ===========================================================================

def run_ablation(name: str, cfg_overrides: dict) -> Dict:
    """
    Run LPEU with specific components disabled.

    A1: No staging       — run_ablation("NoStaging",  {"stage1_epochs": 0, "stage2_epochs": 0, "stage3_epochs": 20})
    A2: No EWC            — run_ablation("NoEWC",      {"use_ewc": False})
    A3: No PCGrad         — run_ablation("NoPCGrad",   {"use_pcgrad": False, "pcgrad_boost": 1.0})
    A4: No K-NN anchor    — run_ablation("NoAnchor",   {"w_kd_local": 0.0})
    A5: Standard margin   — run_ablation("MarginZero", {"repel_margin": 0.0})
    A6: No pair repulsion — run_ablation("NoRepulsion",{"w_repel": 0.0})
    A7: No proto-retain   — run_ablation("NoProtoRetain", {"use_proto_retain": False})
    A8: No L_div (Hue C)  — run_ablation("NoDiv",      {"w_div": 0.0})
    """
    cfg_abl = copy.deepcopy(cfg)
    for k, v in cfg_overrides.items():
        setattr(cfg_abl, k, v)

    anc_loader_abl = None
    if getattr(cfg_abl, "w_kd_local", 0) > 0:
        anc_grp_abl = find_knn_retain_groups(
            old_forget_proto, group_prototypes,
            split["forget_ids"], cfg_abl.k_anchor)   # VGGFace2: forget_ids == group id
        anc_loader_abl = build_local_anchor_loader_grouped(
            eval_base, split["retain_train_idx"], identity_group_ids,
            anc_grp_abl, cfg.batch_size, cfg.num_workers)

    fisher_abl = fisher if cfg_abl.use_ewc else None
    theta_abl  = theta_star if cfg_abl.use_ewc else None

    model_abl = run_lpeu_v7_unlearning(
        model              = clone_model(original_model),
        old_model          = clone_model(original_model),
        forget_loader      = forget_train_loader,
        retain_loader      = retain_train_loader,
        retain_test_loader = retain_test_loader,
        anchor_loader      = anc_loader_abl,
        forget_ids         = split["forget_ids"],
        num_classes        = num_classes,
        cfg                = cfg_abl,
        retain_acc_ref     = retain_acc_ref,
        fisher             = fisher_abl,
        theta_star         = theta_abl,
        retain_prototypes  = all_prototypes,
        local_retain_prototypes = local_retain_prototypes,
        unseen_pool_images      = unseen_pool_images,
    )
    return full_evaluate(
        f"Ablation:{name}", model_abl,
        original_model, retain_test_loader, forget_eval_loader,
        local_neighbor_loader, old_forget_proto, cfg_abl, num_classes,
    )


# Uncomment to run ablations:
# ablation_results = {}
# ablation_results["NoStaging"]   = run_ablation("NoStaging",  {"stage1_epochs": 0, "stage2_epochs": 0, "stage3_epochs": 20})
# ablation_results["NoEWC"]       = run_ablation("NoEWC",      {"use_ewc": False})
# ablation_results["NoPCGrad"]    = run_ablation("NoPCGrad",   {"use_pcgrad": False, "pcgrad_boost": 1.0})
# ablation_results["NoAnchor"]    = run_ablation("NoAnchor",   {"w_kd_local": 0.0})
# ablation_results["MarginZero"]  = run_ablation("MarginZero", {"repel_margin": 0.0})
# ablation_results["NoRepulsion"] = run_ablation("NoRepulsion",{"w_repel": 0.0})
# ablation_results["NoDiv"]       = run_ablation("NoDiv",      {"w_div": 0.0})
# with open(os.path.join(cfg.output_dir, "ablation_vggface2_C.json"), "w") as f:
#     json.dump(ablation_results, f, indent=2)


In [ ]:
# ===========================================================================
# ▓  CELL N: TUNING GUIDE
# ===========================================================================
"""
FILE NÀY LÀ GÌ:
  Port của lpeu-v7-mufac-person-C.ipynb (Hướng C: Migrate→Shatter,
  checkpoint-selection combined-balance, gradient projection, L_div) sang
  VGGFace2, giữ nguyên scale gốc N_IDENTITIES=1500, num_forget_ids TĂNG lên
  100 (từ 30) để khớp SỐ LƯỢNG identity bị quên với MUFAC Hướng C — dùng để
  viết chung 1 bài paper. CHƯA qua vòng tune thực nghiệm nào trên VGGFace2
  (khác các file MUFAC, đã qua v7.1–v7.9 dựa trên log thật) — đọc kỹ log
  epoch ĐẦU TIÊN trước khi tin tưởng số liệu cuối cùng.

KHÁC BIỆT CẤU TRÚC QUAN TRỌNG NHẤT SO VỚI MUFAC (đọc trước khi diễn giải
kết quả):
  MUFAC:     identity ≠ classifier class (age, 8 lớp) → identity_equals_class
             =False → TOÀN BỘ đầu ArcFace bị đóng băng mỗi forget-step →
             forget_accuracy KHÔNG THỂ đổi (tính chất kiến trúc).
  VGGFace2:  identity CHÍNH LÀ classifier class → identity_equals_class=True
             → chỉ hàng ArcFace RETAIN bị đóng băng, hàng FORGET được tự do
             thay đổi → forget_accuracy CÓ THỂ giảm thật, và Δforget_accuracy
             (chứ không chỉ cluster_sim_drop) là một tín hiệu forget hợp lệ
             ở đây — khác hẳn kỳ vọng khi đọc log MUFAC.
  → w_kl_uniform / acc_forget_score trong checkpoint balance CÓ Ý NGHĨA THẬT
    ngay từ lần chạy đầu tiên trên VGGFace2 (không chỉ là "thử xem có tác
    dụng không" như ở MUFAC Hướng B).

HƯỚNG C (L_div) — KHÔNG class-matched ở đây (khác MUFAC):
  D_unseen của VGGFace2 gồm các identity TÁCH BIỆT HOÀN TOÀN với identity
  forget/retain — không có nhãn thô chung (như age-class ở MUFAC) để khớp
  lớp. _sample_unseen_batch_vgg (Cell I) lấy ngẫu nhiên từ TOÀN BỘ D_unseen
  làm target KL, không phân biệt danh tính. Nếu kết quả FS không cải thiện,
  đây là một giả thuyết CẦN xét: việc mất tính "class-matched" có thể làm
  yếu tín hiệu L_div so với bản MUFAC.

HOW TO READ EPOCH LOGS:

Log format:
  [VGGFace2][03/31][ERASE/MIGRATE]  erase=0.721  repel=0.000  kl_uni=0.000
                      div=0.000  w_erase=2.000  w_repel=0.000  w_kl_uni=0.000
                      w_div=0.000  anc=0.0000  ewc=0.0231
                      proto_sim=0.601(drop:+0.150)  H=5.61/5.70
                      fgt_acc=0.0031  ret_acc=0.9048  conf=3.1  boost=40×  anchor_w=0.0

KEY SIGNALS:
  MIGRATE sub-phase:
    → w_repel/w_kl_uni/w_div PHẢI = 0 (chưa sang SHATTER)
    → proto_sim SHOULD DROP each epoch (tín hiệu chính)
    → Chuyển sang SHATTER khi proto_drop đạt migrate_target_fraction (35%)
      của mức giảm lý thuyết tối đa, hoặc epoch ≥ migrate_max_epochs (10)

  SHATTER sub-phase:
    → w_repel/w_kl_uni/w_div PHẢI > 0 (đã sang SHATTER, 1 chiều — không quay
      lại MIGRATE)
    → div SHOULD giảm dần nếu L_div đang có tác dụng (p_forget → p_orig_unseen)
    → fgt_acc CÓ THỂ giảm thật ở VGGFace2 (identity_equals_class=True) — nếu
      KHÔNG giảm dù w_kl_uniform=20 đã full lực, xem xét tăng thêm hoặc kiểm
      tra checkpoint_retain_gate có đang bỏ qua tín hiệu forget hay không.

  Phase ERASE/REFINE/STABLE (lịch anchor, tách biệt khỏi MIGRATE/SHATTER):
    → anc = 0.0000 ở ERASE, > 0 từ REFINE trở đi

QUICK TUNING DECISION TREE:
  proto_sim NOT dropping trong MIGRATE
    → raise pcgrad_boost (40→80→150) OR lower w_ewc OR raise w_erase

  ret_acc drops below 80% sớm
    → raise w_ewc, hoặc raise retain_acc_floor để safety-stop sớm hơn

  intra_cluster_sim starts at 0.90+
    → DELETE original_model.pt và train lại (embedding collapse) — đảm bảo
      dùng FaceModel Simple Neck (không phải BN-Neck)

  cluster_sim_drop < 0.10 sau khi đã vào SHATTER
    → Raise w_repel, hoặc repel_margin âm hơn (-0.3 → -0.5)

  forgetting_score_unseen_paper KHÔNG cải thiện dù w_div=20 full lực trong
  SHATTER
    → Kiểm tra "div=" trong log có thực sự giảm không (nếu không giảm, L_div
      chưa hội tụ — thử tăng w_div hoặc kéo dài SHATTER)
    → Cân nhắc giả thuyết "không class-matched" ở trên — đây là khác biệt
      CÓ CHỦ Ý so với MUFAC, không phải bug, nhưng có thể làm tín hiệu yếu
      hơn

  RAM/OOM khi build unseen_pool_images (Cell L, Step 2)
    → giảm cfg.div_unseen_pool_max (mặc định 2000)
"""

print("\n[LPEU-VGGFace2 Hướng C] All cells ready.")
print(f"  ⚠ CHƯA qua vòng tune thực nghiệm trên VGGFace2 — đọc kỹ log epoch")
print(f"     đầu tiên (MIGRATE→SHATTER transition, div=...) trước khi tin số liệu cuối.")
print(f"  ⚠ Xoá cache cũ nếu đổi kiến trúc model:")
print(f"     import os; os.remove('{os.path.join(cfg.output_dir, 'original_model.pt')}')")
